# Audio Analyser → Visual Prompt → seed42 Control API

This notebook listens to music, describes it, and turns that description into a short visual prompt for the seed42 SDXL-Turbo stream.
It runs **the same code as the Docker server** (commit `4d85c6a` plus uncommitted server changes): the server files are copied unchanged into §2 and imported.

**Pipeline:** audio → features (tempo, energy, genre, mood, lyric themes) → input sentence → **scene decision** → LLM (only when the scene changes) → 40-word prompt → Control API

| Tool | What it provides |
|---|---|
| Librosa | Energy (RMS), fallback tempo, frequency bands |
| Madmom | Neural beat tracking → BPM |
| Essentia | Genre (Discogs 400) and mood (happy, sad, relaxed, aggressive, party) |
| Faster-Whisper | Lyric keywords ("themes") |
| Scene tracker | Compares each window with the music the current scene was written for: **hold**, **evolve**, **blend**, **new scene** or **revert** |
| Groq (`openai/gpt-oss-120b`) | Writes or revises the visual prompt |
| Control API client | Logs in, creates a stream, sends prompts; retries only "not ready" |

### Before you run
1. **Runtime:** Python **3.12 or 3.13** (§1 checks; Docker uses 3.12). GPU is optional and only speeds up Whisper.
2. **Audio:** upload your tracks to `/content` (Files panel) and list them in **§1.2 Configuration**.
3. **LLM key:** add your Groq key in 🔑 **Secrets** under the name `seedAudio` and turn on notebook access. Without a key everything except the LLM step still runs.

Then use **Runtime → Restart session and run all**. Sections run top to bottom.

> **Re-syncing with the server:** when a server file changes, paste its new content into the matching §2 cell, keeping the `%%writefile` line.

---
## 1. Setup
### 1.1 Install packages
The same pinned versions as the Docker image, installed the same way.

In [1]:
import os, sys, pathlib
# The pinned packages have builds for Python 3.12 (what the Docker image uses) and 3.13 (Colab's default)
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise SystemExit(f"Python {sys.version.split()[0]}: the pinned packages need 3.12 or 3.13. "
                     "Change the runtime version, then run all again.")
for d in ("server/app", "mock", "eval/tracks", "eval/results"):
    pathlib.Path("/content/seed42", d).mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0],
      "" if sys.version_info[:2] == (3, 12) else "(the Docker image uses 3.12; note this when reporting versions)")

Python 3.13.16 (the Docker image uses 3.12; note this when reporting versions)


In [2]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null && echo ffmpeg ready

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg ready


In [3]:
%%writefile /content/seed42/server/requirements-build.txt
# Installed first: Madmom compiles against these and is built without
# build isolation, so they must already be in the environment.
numpy==2.5.3
Cython==3.3.0
setuptools==80.9.0
wheel==0.48.0

Overwriting /content/seed42/server/requirements-build.txt


In [4]:
%%writefile /content/seed42/server/requirements.txt
# Tested together on Python 3.12 (Linux x86_64). essentia-tensorflow 2.1b6.dev1389
# also ships macOS arm64 and x86_64 wheels for 3.12; there are no Linux arm64 wheels.
# Install requirements-build.txt first, then this file with --no-build-isolation.

# Server
fastapi==0.142.2
uvicorn[standard]==0.54.0
python-multipart==0.0.32
python-dotenv==1.2.3
requests==2.34.2

# Audio and models
librosa==1.0.0
soundfile==0.14.0
essentia-tensorflow==2.1b6.dev1389
faster-whisper==1.2.1
ctranslate2==4.8.2
madmom @ git+https://github.com/CPJKU/madmom.git@27f032e8947204902c675e5e341a3faf5dc86dae

# LLM (Groq through the OpenAI SDK)
openai==3.22.1

Overwriting /content/seed42/server/requirements.txt


In [5]:
!pip install -q -r /content/seed42/server/requirements-build.txt
!pip install -q --no-build-isolation -r /content/seed42/server/requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [6]:
import importlib.metadata as md
for pkg in ("numpy", "librosa", "soundfile", "essentia-tensorflow", "madmom", "faster-whisper", "openai", "requests"):
    try:
        print(f"{pkg:22s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:22s} not installed")

numpy                  2.2.6
librosa                1.0.0
soundfile              0.14.0
essentia-tensorflow    2.1b6.dev1389
madmom                 0.17.dev0
faster-whisper         1.2.1
openai                 3.22.1
requests               2.34.2


### 1.2 Configuration
Everything you may need to change lives in this cell. The other settings (scene thresholds, window and step,
Control API login) come from `.env.example` in §3.

In [7]:
# --- Audio -------------------------------------------------------------------
# Upload tracks to AUDIO_DIR and list them here. Missing files are skipped.
AUDIO_DIR = "/content"
AUDIO_PLAYLIST = [
    "/content/003-2235411-Jackson Frost-Fashionable Progressive House.mp3",
    "/content/034-1978373-Manfred Merkel-Brighter Than The Sky.mp3",
    "/content/046-1875274-Axined _amp_ Mr. Moohman-How Do I Get My Grass Really Green.mp3",
]

# --- LLM -------------------------------------------------------------------------
GROQ_SECRET_NAME = "seedAudio"     # name of the Colab secret holding your Groq key

# --- Mock Control API ----------------------------------------------------------
# Failure-mode flags for seed42's mock, e.g. ["--fail-rate", "0.2"] or
# ["--not-ready", "409", "--ready-after", "20"]. Empty = normal behaviour.
MOCK_FLAGS = []

print("Configuration set.")

Configuration set.


---
## 2. Server code (exact copies)
One cell per file, in the order audio flows through them. Don't edit these: change the server, then copy the file here.

**`server/app/__init__.py`**: package marker

In [8]:
%%writefile /content/seed42/server/app/__init__.py

Overwriting /content/seed42/server/app/__init__.py


**`server/app/config.py`**: settings, read from environment variables

In [9]:
%%writefile /content/seed42/server/app/config.py
"""All settings, read once at startup.

Each value comes from an environment variable of the same name, or from the
.env file at the project root (see .env.example), and falls back to the
default shown here. Docker Compose passes .env to the container.
"""
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path(__file__).resolve().parents[2] / ".env")   # project root
    load_dotenv()                                                # or the current folder
except ImportError:  # python-dotenv is optional; plain environment variables still work
    pass


# --- Pipeline mode ---------------------------------------------------------
# full = Librosa + Madmom + Essentia + Whisper
# lite = Librosa only; genre, mood and lyric themes are skipped. Starts in
#        seconds, for working on the page or the Control API path.
PIPELINE_MODE = os.getenv("PIPELINE_MODE", "full").strip().lower()
LITE = PIPELINE_MODE == "lite"

# --- Audio and models ------------------------------------------------------
SR = 22050              # every track is resampled to this rate for analysis
ESSENTIA_SR = 16000     # rate the Essentia and Whisper models expect

MODEL_DIR = os.getenv("MODEL_DIR", str(Path(__file__).resolve().parents[1] / "essentia_models"))
WHISPER_SIZE = os.getenv("WHISPER_SIZE", "tiny")     # tiny, base, small, ...
N_THEMES = int(os.getenv("N_THEMES", "4"))           # lyric keywords kept per window
# --- LLM: writes the image prompt (Groq, through the OpenAI SDK) ------------
# Without a key every window uses the fallback prompt in prompt.py.
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
LLM_MODEL = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")
MAX_PROMPT_WORDS = int(os.getenv("MAX_PROMPT_WORDS", "40"))

# --- seed42 Control API: receives the prompts --------------------------------
# Defaults point at the mock in mock/. For the real API set the URL
# (https://seed42.app) and your own seed42 email and password.
CONTROL_API_URL = os.getenv("CONTROL_API_URL", "http://127.0.0.1:8787")
CONTROL_API_EMAIL = os.getenv("CONTROL_API_EMAIL", "test@example.com")
CONTROL_API_PASSWORD = os.getenv("CONTROL_API_PASSWORD", "anything")
CONTROL_API_INITIAL_PROMPT = os.getenv("CONTROL_API_INITIAL_PROMPT", "abstract shapes, vivid, neon")
# Empty: create our own stream (development). Set: send to this stream, the one
# seed42 creates for a performance, and never create or end a stream.
CONTROL_API_STREAM_ID = os.getenv("CONTROL_API_STREAM_ID", "").strip()

# --- Scenes (scene.py) -------------------------------------------------------
# Each window gets a change score from 0 (same music) to 1 (different in every way).
SCENE_EVOLVE_AT = float(os.getenv("SCENE_EVOLVE_AT", "0.25"))      # at or above: adjust the scene
SCENE_CHANGE_AT = float(os.getenv("SCENE_CHANGE_AT", "0.5"))       # at or above: blend, then replace if it lasts
SCENE_GAP_SEC = float(os.getenv("SCENE_GAP_SEC", "1.5"))           # near silence this long suggests a track change
SCENE_SILENCE_RMS = float(os.getenv("SCENE_SILENCE_RMS", "0.01"))  # window energy under this counts as silence

# --- Test page ---------------------------------------------------------------
DEFAULT_WINDOW_SEC = float(os.getenv("DEFAULT_WINDOW_SEC", "20"))  # what the window and step boxes start at
DEFAULT_STEP_SEC = float(os.getenv("DEFAULT_STEP_SEC", "10"))
MAX_UPLOAD_MB = int(os.getenv("MAX_UPLOAD_MB", "200"))

Overwriting /content/seed42/server/app/config.py


**`server/app/models.py`**: loads Madmom, Essentia and Whisper once

In [10]:
%%writefile /content/seed42/server/app/models.py
"""Loads every model once and keeps it in memory.

Each component loads independently and reports its own status, so a failed
download disables one feature instead of the whole pipeline: genre and mood
become "unavailable", or tempo falls back to Librosa.
"""
import collections
import collections.abc
import json
import logging
import os
import threading
import time
import urllib.request
import warnings

import numpy as np

from . import config

log = logging.getLogger("pipeline.models")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# --- NumPy / collections shims (must run before importing madmom) ---------
for _name, _py in {"float": float, "int": int, "complex": complex,
                   "object": object, "bool": bool, "str": str}.items():
    if not hasattr(np, _name):
        setattr(np, _name, _py)
for _name, _val in {"NINF": -np.inf, "PINF": np.inf, "Inf": np.inf,
                    "float_": np.float64, "complex_": np.complex128}.items():
    if not hasattr(np, _name):
        setattr(np, _name, _val)
for _abc in ("MutableSequence", "Iterable", "Callable", "MutableMapping", "Mapping", "Sequence"):
    if not hasattr(collections, _abc):
        setattr(collections, _abc, getattr(collections.abc, _abc))

MODEL_BASE = "https://essentia.upf.edu/models"
MOOD_NAMES = ["happy", "sad", "relaxed", "aggressive", "party"]
STOPWORDS = set("a an the and or but if then than that this in on into to for with "
                "without about oh yeah uh ah hey".split())


class Models:
    """Holds the loaded models plus a status line per component for /api/health."""

    def __init__(self):
        self.status = {
            "madmom": {"state": "pending", "detail": ""},
            "essentia": {"state": "pending", "detail": ""},
            "mood": {"state": "pending", "detail": ""},
            "whisper": {"state": "pending", "detail": ""},
        }
        self.ready = threading.Event()
        self.device = "cpu"
        # Filled by load()
        self.RNNBeatProcessor = None
        self.DBNBeatTrackingProcessor = None
        self.embedding_model = None
        self.genre_head = None
        self.genre_classes = []
        self.mood_heads = {}        # {mood: (head, index of "yes")}
        self.whisper = None

    # -- helpers -----------------------------------------------------------
    def _set(self, key, state, detail=""):
        self.status[key] = {"state": state, "detail": detail}
        log.info("%-8s %s %s", key, state, detail)

    @property
    def madmom_ok(self):
        return self.status["madmom"]["state"] == "ready"

    @property
    def essentia_ok(self):
        return self.status["essentia"]["state"] == "ready"

    @property
    def mood_ok(self):
        return self.status["mood"]["state"] == "ready"

    @property
    def whisper_ok(self):
        return self.status["whisper"]["state"] == "ready"

    def load_in_background(self):
        threading.Thread(target=self.load, name="model-loader", daemon=True).start()

    # -- loading -----------------------------------------------------------
    def load(self):
        t0 = time.time()
        if config.LITE:
            for k in self.status:
                self._set(k, "skipped", "lite mode")
            self.ready.set()
            return

        try:
            import ctranslate2
            if ctranslate2.get_cuda_device_count() > 0:
                self.device = "cuda"
        except Exception:
            pass

        self._load_madmom()
        self._load_essentia()
        self._load_whisper()
        log.info("models loaded in %.1fs on %s", time.time() - t0, self.device)
        self.ready.set()

    def _load_madmom(self):
        self._set("madmom", "loading")
        try:
            from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor
            self.RNNBeatProcessor = RNNBeatProcessor
            self.DBNBeatTrackingProcessor = DBNBeatTrackingProcessor
            self._set("madmom", "ready")
        except Exception as e:
            self._set("madmom", "failed", f"falling back to Librosa tempo: {e}")

    def _download_essentia(self):
        os.makedirs(config.MODEL_DIR, exist_ok=True)
        files = {
            "embeddings": ("feature-extractors/discogs-effnet", "discogs-effnet-bs64-1.pb"),
            "genre": ("classification-heads/genre_discogs400", "genre_discogs400-discogs-effnet-1.pb"),
            "genre_meta": ("classification-heads/genre_discogs400", "genre_discogs400-discogs-effnet-1.json"),
        }
        for m in MOOD_NAMES:
            for ext in (".pb", ".json"):
                files[f"mood_{m}{ext}"] = (f"classification-heads/mood_{m}", f"mood_{m}-discogs-effnet-1{ext}")
        for subdir, fname in files.values():
            dest = os.path.join(config.MODEL_DIR, fname)
            if not os.path.exists(dest):
                log.info("downloading %s", fname)
                tmp = dest + ".part"
                urllib.request.urlretrieve(f"{MODEL_BASE}/{subdir}/{fname}", tmp)
                os.replace(tmp, dest)   # no half-written model files left behind

    def _load_essentia(self):
        self._set("essentia", "loading", "downloading models on first run")
        self._set("mood", "loading")
        try:
            from essentia.standard import TensorflowPredictEffnetDiscogs, TensorflowPredict2D
            self._download_essentia()
            d = config.MODEL_DIR
            self.embedding_model = TensorflowPredictEffnetDiscogs(
                graphFilename=os.path.join(d, "discogs-effnet-bs64-1.pb"),
                output="PartitionedCall:1",
            )
            with open(os.path.join(d, "genre_discogs400-discogs-effnet-1.json")) as f:
                self.genre_classes = json.load(f)["classes"]
            self.genre_head = TensorflowPredict2D(
                graphFilename=os.path.join(d, "genre_discogs400-discogs-effnet-1.pb"),
                input="serving_default_model_Placeholder",
                output="PartitionedCall",
            )
            self._set("essentia", "ready", f"{len(self.genre_classes)} genres")
        except Exception as e:
            self._set("essentia", "failed", f"genre and mood unavailable: {e}")
            self._set("mood", "failed", "needs the Essentia embedding model")
            return

        # Mood heads load separately so a mood failure never disables genre
        skipped = []
        for m in MOOD_NAMES:
            try:
                head, classes = self._mood_head(TensorflowPredict2D, f"mood_{m}-discogs-effnet-1")
                self.mood_heads[m] = (head, classes.index(m))
            except Exception as e:
                skipped.append(f"{m} ({e})")
        if self.mood_heads:
            self._set("mood", "ready", ", ".join(self.mood_heads) +
                      (f"; skipped {', '.join(skipped)}" if skipped else ""))
        else:
            self._set("mood", "failed", "; ".join(skipped))

    @staticmethod
    def _mood_head(TensorflowPredict2D, name):
        """Loads a mood head and returns (head, class names)."""
        d = config.MODEL_DIR
        with open(os.path.join(d, f"{name}.json")) as f:
            meta = json.load(f)
        # Tensor names differ between Essentia releases; read them from the metadata
        schema = meta.get("schema", {})
        inp = schema.get("inputs", [{}])[0].get("name", "model/Placeholder")
        out = next((o["name"] for o in schema.get("outputs", [])
                    if o.get("output_purpose") == "predictions"), "model/Softmax")
        head = TensorflowPredict2D(graphFilename=os.path.join(d, f"{name}.pb"), input=inp, output=out)
        return head, meta["classes"]

    def _load_whisper(self):
        self._set("whisper", "loading", f"{config.WHISPER_SIZE}, downloads on first run")
        try:
            from faster_whisper import WhisperModel
            compute = "float16" if self.device == "cuda" else "int8"
            self.whisper = WhisperModel(config.WHISPER_SIZE, device=self.device, compute_type=compute)
            self._set("whisper", "ready", f"{config.WHISPER_SIZE} on {self.device}")
        except Exception as e:
            self._set("whisper", "failed", f"lyric themes unavailable: {e}")


MODELS = Models()

Overwriting /content/seed42/server/app/models.py


**`server/app/features.py`**: features and the input sentence for one window

In [11]:
%%writefile /content/seed42/server/app/features.py
"""Feature extraction for one window of audio.

Tempo (Madmom, falling back to Librosa), energy and its phase, the loudest
frequency band, genre and mood (Essentia) and lyric keywords (Whisper), plus
the one-sentence description the LLM turns into an image prompt. Each result
carries short machine keys (phase, band) next to the readable labels, and
per-stage timings.
"""
import io
import time
from collections import Counter

import librosa
import numpy as np
import soundfile as sf

from . import config
from .models import MODELS, STOPWORDS

#loudness thresholds for the four energy phases
PHASES = [  # (upper RMS bound, key, label)
    (0.15, "breakdown", "Low Energy / Breakdown Phase"),
    (0.40, "groove", "Moderate Energy / Steady Groove Phase"),
    (0.65, "build", "High Energy / Build-up Phase"),
    (float("inf"), "drop", "Peak Energy / Drop Phase"),
]
 #frequency bands for the three loudest ranges, used to describe the track's focus
BANDS = [  # (key, label, lo Hz, hi Hz)
    ("sub", "Low-End / Sub-Bass Focus (20 Hz - 120 Hz)", 20, 120),
    ("mid", "Mid-Range / Vocal & Synth Focus (120 Hz - 2 kHz)", 120, 2000),
    ("high", "High-End / Percussion & Hi-Hat Focus (2 kHz - 8 kHz)", 2000, 8000),
]


def get_mood(embeddings, threshold=0.5, top_k=2):
    """Up to `top_k` moods above `threshold` confidence, strongest first, or 'neutral';
    plus every score. One yes/no head per mood."""
    scores = {}
    for m, (head, idx) in MODELS.mood_heads.items():
        try:
            scores[m] = float(np.mean(head(embeddings), axis=0)[idx])
        except Exception:
            pass
    picked = sorted(((s, m) for m, s in scores.items() if s >= threshold), reverse=True)[:top_k]
    return (", ".join(m for _, m in picked) if picked else "neutral"), scores


def extract_fast_features(y_slice, sr=config.SR, n_themes=3):
    """Runs Librosa, Madmom, Essentia and Whisper on one audio slice, entirely in RAM."""
    timings = {}

    # --- 1. Librosa: energy and fallback tempo ---------------------------------
    t = time.perf_counter()
    rms = float(np.sqrt(np.mean(y_slice ** 2)))
    librosa_bpm = round(float(np.atleast_1d(
        librosa.feature.tempo(y=y_slice, sr=sr, start_bpm=120))[0]), 1)
    timings["librosa"] = time.perf_counter() - t

    # --- 2. Madmom: neural beat tracking --------------------------------------
    madmom_bpm, beat_count = 0.0, 0
    if MODELS.madmom_ok:
        t = time.perf_counter()
        try:
            wav_io = io.BytesIO()                        # in-memory WAV, no disk writes
            sf.write(wav_io, y_slice, sr, format="WAV")
            wav_io.seek(0)
            beat_times = np.asarray(
                MODELS.DBNBeatTrackingProcessor(fps=100)(MODELS.RNNBeatProcessor()(wav_io)), dtype=float)
            if len(beat_times) >= 2:
                madmom_bpm = round(60.0 / float(np.median(np.diff(beat_times))), 1)
                beat_count = len(beat_times)
        except Exception:
            madmom_bpm = 0.0
        timings["madmom"] = time.perf_counter() - t

    if madmom_bpm > 0:
        bpm, bpm_source = madmom_bpm, "Madmom neural"
    else:
        bpm, bpm_source = librosa_bpm, "Librosa DSP"

    # --- 3. Essentia: genre and mood ------------------------------------------
    y16 = None
    mood_scores = {}
    if MODELS.essentia_ok or MODELS.whisper_ok:
        y16 = librosa.resample(y_slice, orig_sr=sr, target_sr=config.ESSENTIA_SR) if sr != config.ESSENTIA_SR else y_slice
        y16 = np.ascontiguousarray(y16, dtype=np.float32)

    if MODELS.essentia_ok:
        t = time.perf_counter()
        embeddings = MODELS.embedding_model(y16)
        genre_probs = np.mean(MODELS.genre_head(embeddings), axis=0)
        top_idx = np.argsort(genre_probs)[::-1][:2]
        top_genres = [MODELS.genre_classes[i] for i in top_idx]
        genre_scores = [round(float(genre_probs[i]), 3) for i in top_idx]
        mood_str, mood_scores = get_mood(embeddings) if MODELS.mood_ok else ("unavailable", {})
        timings["essentia"] = time.perf_counter() - t
    else:
        top_genres, genre_scores, mood_str = ["unavailable"], [], "unavailable"

    primary_genre = top_genres[0]
    secondary_genre = top_genres[1] if len(top_genres) > 1 else "N/A"
    genres_str = ", ".join(top_genres)

    # --- 4. Whisper: lyric keywords -------------------------------------------
    # Only word counts are kept; the transcript itself is never stored or returned.
    if MODELS.whisper_ok:
        t = time.perf_counter()
        segments, _ = MODELS.whisper.transcribe(y16, beam_size=1, vad_filter=True)
        tokens = []
        for seg in segments:
            if getattr(seg, "no_speech_prob", 0.0) <= 0.6:
                words = "".join(c.lower() if c.isalpha() or c.isspace() else " " for c in seg.text).split()
                tokens.extend(w for w in words if len(w) >= 4 and w not in STOPWORDS)
        top_themes = [w for w, _ in Counter(tokens).most_common(n_themes)]
        themes_str = ", ".join(top_themes) if top_themes else "instrumental"
        timings["whisper"] = time.perf_counter() - t
    else:
        themes_str = "unavailable"

    return {
        "bpm": bpm,
        "bpm_source": bpm_source,
        "madmom_bpm": madmom_bpm,
        "librosa_bpm": librosa_bpm,
        "beat_count": beat_count,
        "energy": round(rms, 4),
        "top_2_genres": genres_str,
        "genre_scores": genre_scores,
        "primary_genre": primary_genre,
        "secondary_genre": secondary_genre,
        "mood": mood_str,
        "mood_scores": {k: round(v, 3) for k, v in mood_scores.items()},
        "lyric_themes": themes_str,
        "timings": {k: round(v * 1000) for k, v in timings.items()},
    }


def get_dominant_frequency_band(y_slice, sr):
    stft = np.abs(librosa.stft(y_slice, n_fft=2048, hop_length=512))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

    def band_energy(lo, hi):
        mask = (freqs > lo) & (freqs <= hi)
        return float(np.mean(stft[mask, :])) if np.any(mask) else 0.0

    energies = {key: band_energy(lo, hi) for key, _, lo, hi in BANDS}
    key = max(energies, key=energies.get)
    label = next(lbl for k, lbl, _, _ in BANDS if k == key)
    return key, label


def get_energy_phase(rms):
    for bound, key, label in PHASES:
        if rms < bound:
            return key, label


def get_energy_descriptor(rms):
    return f"{rms:.3f} ({get_energy_phase(rms)[1]})"


def build_llm_input(features, energy_desc, dominant_band):
    """One sentence describing the window, used as the LLM's input.

    When a model is missing (lite mode, a failed download) its part is left
    out instead of sending the LLM the word "unavailable".
    """
    energy_phase = energy_desc.split("(")[1].rstrip(")")
    band_name = dominant_band.split("(")[0].strip()
    mood = features["mood"]
    genre = features["primary_genre"]
    themes = features["lyric_themes"]

    if "unavailable" not in (mood, genre, themes):
        genre_name = genre.split("---")[-1]    # "Electronic---House" -> "House"
        return (f"A {mood} mood, {energy_phase} {features['bpm']} BPM {genre_name} atmosphere, "
                f"featuring {band_name} with {themes} themes.")

    parts = [f"A {mood} mood," if mood != "unavailable" else "A",
             f"{energy_phase} {features['bpm']} BPM"]
    if genre != "unavailable":
        parts.append(genre.split("---")[-1])
    parts.append(f"atmosphere, featuring {band_name}")
    sentence = " ".join(parts)
    if themes != "unavailable":
        sentence += f" with {themes} themes"
    return sentence + "."


def analyse_window(y_slice, sr=config.SR):
    """Everything the test page shows for one window, except the LLM and API steps."""
    t0 = time.perf_counter()
    features = extract_fast_features(y_slice, sr=sr, n_themes=config.N_THEMES)
    phase_key, phase_label = get_energy_phase(features["energy"])
    t = time.perf_counter()
    band_key, band_label = get_dominant_frequency_band(y_slice, sr)
    features["timings"]["bands"] = round((time.perf_counter() - t) * 1000)
    energy_desc = get_energy_descriptor(features["energy"])
    features.update({
        "phase": phase_key,
        "phase_label": phase_label,
        "band": band_key,
        "band_label": band_label,
        "llm_input": build_llm_input(features, energy_desc, band_label),
        "analysis_ms": round((time.perf_counter() - t0) * 1000),
    })
    return features


def energy_envelope(y, sr=config.SR, max_points=1200):
    """Raw RMS over the whole track for the page's energy strip.

    Same RMS scale as the window energy, so the phase thresholds line up.
    """
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    if len(rms) > max_points:
        block = int(np.ceil(len(rms) / max_points))
        pad = (-len(rms)) % block
        rms = np.pad(rms, (0, pad)).reshape(-1, block).mean(axis=1)
        dt = block * hop / sr
    else:
        dt = hop / sr
    return {"values": [round(float(v), 4) for v in rms], "dt": dt}

Overwriting /content/seed42/server/app/features.py


**`server/app/scene.py`**: change score and scene decisions

In [12]:
%%writefile /content/seed42/server/app/scene.py
"""Scenes as gradual, revisable states instead of a new prompt every window.

The first audible window sets the scene. After that each window is compared
with the features the scene was written for, and the scene is:

  hold       little change: the same prompt is re-sent (sending nothing would
             leave the stream without a prompt and the image goes black)
  evolve     moderate change: the LLM revises the current prompt, keeping its
             main scene and changing a few phrases
  blend      large change, or a gap of near silence: the LLM writes a halfway
             prompt between the current scene and the new audio, and the
             change is marked pending
  new scene  the next window confirms the change: a fresh scene is written.
             After a gap (a likely track change) a moderate change is enough.
  revert     the next window doesn't confirm it: back to the scene from before
             the blend

A new uploaded file is a known track boundary: its first window blends from
the previous scene and its second window always starts a new scene.

The prompt can also be set by hand (POST /api/scene). Unlocked, the pipeline
keeps it and evolves from it when the music changes. Locked, every window
re-sends it unchanged.

There is one tracker for the server, like the one stream in control.py.
"""
import logging
import math
import threading
import time

import librosa
import numpy as np

from . import config
from .prompt import blend_visual_prompt, evolve_visual_prompt, generate_visual_prompt

log = logging.getLogger("pipeline.scene")

PHASE_ORDER = ["breakdown", "groove", "build", "drop"]
WEIGHTS = {"bpm": 0.25, "genre": 0.25, "mood": 0.2, "energy": 0.2, "band": 0.1}
REF_KEYS = ("bpm", "energy", "phase", "band", "primary_genre", "secondary_genre", "mood_scores")


# ---------------------------------------------------------------- change score
def _bpm_change(a, b):
    if a <= 0 or b <= 0:
        return None
    d = abs(math.log2(b / a))
    d = min(d, abs(d - 1))              # half or double time counts as the same tempo
    return min(1.0, (2 ** d - 1) / 0.10)  # a 10% tempo change scores 1


def _genre_change(a, b):
    ga, gb = a.get("primary_genre"), b.get("primary_genre")
    if not ga or not gb or "unavailable" in (ga, gb):
        return None
    if ga == gb:
        return 0.0
    if gb == a.get("secondary_genre") or ga == b.get("secondary_genre"):
        return 0.4                      # top two swapped places
    if ga.split("---")[0] == gb.split("---")[0]:
        return 0.6                      # same parent genre
    return 1.0


def _mood_change(a, b):
    ma, mb = a.get("mood_scores") or {}, b.get("mood_scores") or {}
    keys = ma.keys() & mb.keys()
    if not keys:
        return None
    return min(1.0, sum(abs(ma[k] - mb[k]) for k in keys) / len(keys) / 0.25)


def _energy_change(a, b):
    ea, eb = a["energy"], b["energy"]
    rel = abs(eb - ea) / max(ea, eb, 0.05)
    jump = abs(PHASE_ORDER.index(a["phase"]) - PHASE_ORDER.index(b["phase"]))
    return max(min(1.0, rel / 0.6), 1.0 if jump >= 2 else 0.0)


def change_score(ref, rec):
    """0 = same music as the scene was written for, 1 = different in every way.

    Parts that a missing model can't provide (lite mode) are left out and the
    remaining weights are scaled up.
    """
    parts = {
        "bpm": _bpm_change(ref["bpm"], rec["bpm"]),
        "genre": _genre_change(ref, rec),
        "mood": _mood_change(ref, rec),
        "energy": _energy_change(ref, rec),
        "band": 0.0 if ref["band"] == rec["band"] else 1.0,
    }
    parts = {k: round(v, 2) for k, v in parts.items() if v is not None}
    total = sum(WEIGHTS[k] for k in parts)
    score = sum(WEIGHTS[k] * v for k, v in parts.items()) / total
    return round(score, 3), parts


def find_gap(y, sr=config.SR):
    """(start, end) in seconds of the longest near-silent stretch, if it lasts
    at least SCENE_GAP_SEC. Near silent means under a tenth of the window's
    loud level, so it works for quiet microphone input too."""
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    if not len(rms):
        return None
    loud = float(np.percentile(rms, 90))
    if loud < config.SCENE_SILENCE_RMS:
        return None                     # the whole window is quiet; handled as silence
    quiet = np.append(rms < 0.1 * loud, False)
    best, start = (0, 0), None
    for i, q in enumerate(quiet):
        if q and start is None:
            start = i
        elif not q and start is not None:
            if i - start > best[1] - best[0]:
                best = (start, i)
            start = None
    if (best[1] - best[0]) * hop / sr < config.SCENE_GAP_SEC:
        return None
    return best[0] * hop / sr, best[1] * hop / sr


# ---------------------------------------------------------------- tracker
LLM_STEPS = {"initial", "evolve", "blend", "new_scene", "new_track"}


class SceneTracker:
    def __init__(self):
        self.lock = threading.Lock()
        self.track = 0
        self.scene_id = 0
        self._clear()

    def _clear(self):
        self.prompt = None
        self.ref = None             # features the current scene was written for
        self.pending = None         # {"before": prompt, "track": bool, "forced": bool} while a change is unconfirmed
        self.locked = False
        self.manual = False
        self.decision = None
        self.updated = None
        self.last_gap_end = -1.0

    def describe(self):
        return {"prompt": self.prompt, "scene": self.scene_id, "track": self.track,
                "locked": self.locked, "manual": self.manual, "pending": self.pending is not None,
                "decision": self.decision, "updated": self.updated,
                "thresholds": {"evolve": config.SCENE_EVOLVE_AT, "change": config.SCENE_CHANGE_AT}}

    def current_prompt(self):
        with self.lock:
            return self.prompt

    def start_track(self, source):
        """A known track boundary: a new upload, or the start of a live session."""
        with self.lock:
            self.track += 1
            self.ref = None
            self.last_gap_end = -1.0
            self.pending = None
            if self.prompt and not self.locked:
                # First window blends from the previous scene, second one replaces it
                self.pending = {"before": self.prompt, "track": True, "forced": True, "boundary": True}
            log.info("track %d: %s", self.track, source)

    def set_manual(self, prompt=None, locked=None):
        with self.lock:
            if prompt is not None:
                if prompt != self.prompt:
                    self.scene_id += 1
                self.prompt = prompt
                self.manual = True
                self.pending = None
                self.ref = None             # the next window becomes its reference
                self.decision = "manual"
                self.updated = time.time()
            if locked is not None:
                if self.locked and not locked:
                    self.ref = None         # resume from the music playing now, not from before the lock
                self.locked = bool(locked) and self.prompt is not None
            return self.describe()

    def step(self, rec, gap, use_llm):
        """Decides what this window sends, fills in the prompt fields of `rec`
        and returns the prompt (None = send nothing)."""
        with self.lock:
            score, parts = change_score(self.ref, rec) if self.ref else (None, {})
            gap_at = None
            if gap and rec["t_start"] + gap[0] >= self.last_gap_end:   # an overlapping window sees the same gap again
                self.last_gap_end = rec["t_start"] + gap[1]
                gap_at = round(rec["t_start"] + gap[0], 2)

            decision, note = self._decide(rec, score, gap_at)
            if decision in LLM_STEPS and not use_llm:
                decision, note = ("hold", "Prompt generation is off, so the scene is held.") if self.prompt \
                    else ("wait", "Prompt generation is off and there's no scene yet.")
                self.pending = None

            llm = {"prompt": None, "source": None, "ms": 0, "error": None}
            if decision in ("initial", "new_scene", "new_track"):
                llm = generate_visual_prompt(rec["llm_input"])
            elif decision == "evolve":
                llm = evolve_visual_prompt(self.prompt, rec["llm_input"])
            elif decision == "blend":
                llm = blend_visual_prompt(self.prompt, rec["llm_input"])

            if decision in LLM_STEPS and llm["source"] != "llm" and self.prompt:
                # The LLM failed: keep what's on screen rather than jump to the fallback prompt
                note = f"{llm['error']} Holding the current scene instead."
                decision, llm = "hold", {**llm, "prompt": None, "source": None}
                self.pending = None
            self._commit(decision, rec, llm["prompt"])

            source = {"hold": "held", "revert": "held", "locked": "manual",
                      "wait": "off" if not use_llm else "waiting"}.get(decision, llm["source"])
            rec.update({"prompt": self.prompt if decision != "wait" else None, "prompt_source": source,
                        "llm_ms": llm["ms"], "llm_error": llm["error"]})
            rec["scene"] = {"decision": decision, "note": note, "score": score, "parts": parts,
                            "gap_at": gap_at, **self.describe()}
            return rec["prompt"]

    def _decide(self, rec, score, gap_at):
        evolve_at, change_at = config.SCENE_EVOLVE_AT, config.SCENE_CHANGE_AT
        if self.locked:
            return "locked", "Your prompt is locked and re-sent every window."
        if rec["energy"] < config.SCENE_SILENCE_RMS:
            if not self.prompt:
                return "wait", "Near silence. Waiting for audio before choosing the first scene."
            if not self.pending:
                self.pending = {"before": self.prompt, "track": True, "forced": False}
            return "hold", "Near silence, so a track may be changing. Holding the scene."
        if not self.prompt:
            return "initial", "First audible window, so it sets the scene."

        p = self.pending
        if p:
            self.pending = None
            if p.get("boundary"):
                if p.get("blended"):
                    return "new_track", "New track. The scene that blended in is now replaced with one written for it."
                p["blended"] = True
                self.pending = p
                return "blend", "New track. Blending from the previous track's scene before replacing it."
            threshold = evolve_at if p["track"] else change_at
            if p["forced"] or score is None or score >= threshold:
                if p["track"]:
                    return "new_track", f"Change {score} after a gap confirms a new track (threshold {threshold})."
                return "new_scene", f"Change {score} held for a second window, so the scene is replaced."
            if self.prompt != p["before"]:
                self.prompt = p["before"]
                return "revert", f"Change fell to {score}, under {threshold}. Back to the scene from before the blend."
            return "hold", f"Change {score} after the gap is under {threshold}. Same track, scene held."

        if score is None:
            # A prompt set by hand has no features yet; this window becomes its reference
            self.ref = {k: rec[k] for k in REF_KEYS}
            return "hold", "This window becomes the reference for your prompt. Later windows are compared with it."
        if gap_at is not None:
            self.pending = {"before": self.prompt, "track": True, "forced": False}
            return "blend", f"Gap of near silence at {gap_at:.1f} s. Blending toward the new audio until the next window confirms."
        if score >= change_at:
            self.pending = {"before": self.prompt, "track": False, "forced": False}
            return "blend", f"Change {score} is over {change_at}. Blending; the next window confirms or reverts."
        if score >= evolve_at:
            return "evolve", f"Change {score} is between {evolve_at} and {change_at}, so the scene is adjusted."
        return "hold", f"Change {score} is under {evolve_at}. Same prompt re-sent."

    def _commit(self, decision, rec, new_prompt):
        if new_prompt:
            self.prompt = new_prompt
            self.manual = False
            self.updated = time.time()
        if decision in ("initial", "new_scene", "new_track"):
            self.scene_id += 1
        if decision in ("initial", "new_scene", "new_track", "evolve") or (decision == "blend" and self.ref is None):
            self.ref = {k: rec[k] for k in REF_KEYS}
        self.decision = decision


SCENE = SceneTracker()

Overwriting /content/seed42/server/app/scene.py


**`server/app/prompt.py`**: Groq prompt writing: new, evolve, blend

In [13]:
%%writefile /content/seed42/server/app/prompt.py
"""Audio description -> short SDXL-Turbo prompt, written by an LLM on Groq.

Needs GROQ_API_KEY. Without a key the pipeline still runs and uses
FALLBACK_PROMPT, so the Control API path can be tested anyway.
"""
import logging
import time

from . import config

log = logging.getLogger("pipeline.llm")

FALLBACK_PROMPT = "dark club space, sub-bass pulse, neon haze, slow camera drift"

SYSTEM_PROMPT = (
    "You write prompts for a real-time SDXL-Turbo image stream. "
    "Turn the audio cue into ONE line of 8 to 12 short visual phrases separated by commas, "
    f"{config.MAX_PROMPT_WORDS} words maximum.\n"
    "Order: main scene first, then surface materials, lighting colours, atmosphere "
    "(fog, particles, air movement), glowing elements, and camera motion last.\n"
    "RULES: no full sentences, no markdown, no quotes, no numbers or BPM, "
    "no genre names, no words or text to appear in the image."
)

_client = None
if config.GROQ_API_KEY:
    try:
        from openai import OpenAI
        _client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=config.GROQ_API_KEY)
        log.info("Groq client ready (model: %s)", config.LLM_MODEL)
    except Exception as e:
        log.warning("Groq client failed to start: %s", e)


def llm_enabled():
    return _client is not None


def trim_prompt(text, max_words=config.MAX_PROMPT_WORDS):
    """Cleans formatting and cuts to `max_words`, ending on a complete phrase."""
    text = (text.replace("\n", ", ").replace("**", "")
                .replace('"', "").replace("\u2011", "-"))
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words])
        if "," in text:
            text = text.rsplit(",", 1)[0]
    return text.strip(" ,.")


EVOLVE_PROMPT = SYSTEM_PROMPT + (
    "\nYou are revising the scene already on screen, not replacing it. Keep its main scene "
    "and most of its phrases. Change at most three phrases, preferring lighting colours, "
    "atmosphere, glowing elements and camera motion, so it fits the new audio cue. "
    "Return the whole revised line."
)

BLEND_PROMPT = SYSTEM_PROMPT + (
    "\nThe music is changing. Write a transitional line halfway between the current scene "
    "and the new audio cue: keep the setting of the current scene, and bring in the "
    "lighting, atmosphere and motion the new cue suggests, so the image can morph toward "
    "a new scene."
)


def generate_visual_prompt(llm_input):
    """A fresh scene. Returns {prompt, source, ms, error}; source is 'llm', 'fallback' or 'empty-reply'."""
    return _complete(SYSTEM_PROMPT, f"Audio cue: {llm_input}", 0.7)


def evolve_visual_prompt(current, llm_input):
    """The current scene, adjusted a little for the new audio."""
    return _complete(EVOLVE_PROMPT, f"Current scene: {current}\nNew audio cue: {llm_input}", 0.4)


def blend_visual_prompt(current, llm_input):
    """Halfway between the current scene and the new audio."""
    return _complete(BLEND_PROMPT, f"Current scene: {current}\nNew audio cue: {llm_input}", 0.6)


def _complete(system, user, temperature):
    if _client is None:
        return {"prompt": FALLBACK_PROMPT, "source": "fallback", "ms": 0,
                "error": "GROQ_API_KEY is not set, so the fallback prompt was used."}
    t = time.perf_counter()
    try:
        reply = _client.chat.completions.create(
            model=config.LLM_MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            temperature=temperature,
            max_tokens=400,                           # gpt-oss counts its reasoning in this budget
            extra_body={"reasoning_effort": "low"},   # keeps reasoning short so the answer isn't cut off
        )
        text = (reply.choices[0].message.content or "").strip()
        ms = round((time.perf_counter() - t) * 1000)
        if not text:
            return {"prompt": FALLBACK_PROMPT, "source": "empty-reply", "ms": ms,
                    "error": "The LLM returned an empty reply, so the fallback prompt was used."}
        return {"prompt": trim_prompt(text), "source": "llm", "ms": ms, "error": None}
    except Exception as e:
        return {"prompt": FALLBACK_PROMPT, "source": "fallback",
                "ms": round((time.perf_counter() - t) * 1000),
                "error": f"LLM call failed: {e}"}

Overwriting /content/seed42/server/app/prompt.py


**`server/app/control.py`**: seed42 Control API client

In [14]:
%%writefile /content/seed42/server/app/control.py
"""seed42 Control API client: log in, create a stream, send prompts.

Every call reports what happened -- status code, attempts, latency -- instead
of raising, because showing that on the test page is the point.

Follows the client's control-api.md: PATCH /api/streams with {id, params}, and
the request discipline in its section 4 (one request in flight, last write
wins, 12 s timeout, retry only "not ready").
"""
import logging
import threading
import time

import requests

from . import config

log = logging.getLogger("pipeline.control")


def is_not_ready(code, body):
    """The one retryable answer: a 409, or a 404 whose body says not ready.
    Any other 404 means the stream doesn't exist, and retrying won't help."""
    return code == 409 or (code == 404 and "not ready" in str(body).lower())


class Seed42Client:
    def __init__(self, base_url, timeout_seconds=12):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout_seconds
        self.token = None

    def login(self, email, password):
        r = requests.post(f"{self.base_url}/api/auth",
                          json={"action": "login", "email": email, "password": password},
                          timeout=self.timeout)
        r.raise_for_status()
        self.token = r.json()["token"]
        return self.token

    def _headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def create_stream(self, initial_prompt):
        r = requests.post(f"{self.base_url}/api/streams", headers=self._headers(),
                          json={"params": {"model_id": "stabilityai/sdxl-turbo",
                                           "width": 1024, "height": 576, "prompt": initial_prompt}},
                          timeout=self.timeout)
        r.raise_for_status()
        return r.json()["id"], r.status_code == 202

    def get_status(self, stream_id):
        r = requests.get(f"{self.base_url}/api/streams", headers=self._headers(),
                         params={"id": stream_id}, timeout=self.timeout)
        if r.status_code == 404:
            return "not_found"
        r.raise_for_status()
        return r.json().get("status")

    def wait_until_ready(self, stream_id, poll_every=2, max_wait=30):
        waited = 0
        while waited < max_wait:
            if self.get_status(stream_id) == "ready":
                return True
            time.sleep(poll_every)
            waited += poll_every
        return False

    def send_prompt(self, stream_id, prompt, max_attempts=3, retry_wait=2):
        """Returns {code, attempts, ms, body}. Only "not ready" is retried."""
        t0 = time.perf_counter()
        attempt = 0
        while True:
            attempt += 1
            # Only the prompt -- a hot field. Never send creation-time fields here.
            r = requests.patch(f"{self.base_url}/api/streams", headers=self._headers(),
                               json={"id": stream_id, "params": {"prompt": prompt}},
                               timeout=self.timeout)
            try:
                body = r.json()
            except ValueError:
                body = {"raw": r.text[:200]}
            if is_not_ready(r.status_code, body) and attempt < max_attempts:
                time.sleep(retry_wait)
                continue
            return {"code": r.status_code, "attempts": attempt,
                    "ms": round((time.perf_counter() - t0) * 1000), "body": body}

    def end_stream(self, stream_id):
        try:
            r = requests.delete(f"{self.base_url}/api/streams", headers=self._headers(),
                                json={"stream_id": stream_id}, timeout=self.timeout)
            return r.status_code == 200
        except requests.RequestException:
            return False


HINTS = {
    400: "Bad request. Retrying won't help; check what is being sent.",
    401: "Not logged in, usually an expired token. The next prompt logs in again. "
         "If it keeps happening, check CONTROL_API_EMAIL / CONTROL_API_PASSWORD.",
    404: "Still not ready after 3 tries.",
    409: "Still not ready after 3 tries.",
}
NEXT_OWN = "A new stream is created on the next window."
NEXT_GIVEN = ("This app doesn't replace a stream seed42 gave it. "
              "Ask seed42 for the new id and set CONTROL_API_STREAM_ID.")


class StreamLink:
    """One stream for the whole server session.

    With CONTROL_API_STREAM_ID set, that stream is used as given: in a real
    performance seed42's own client creates it, so this never creates or ends
    one. Without it, a stream is created on first use, for development.

    One request is in flight at a time. A prompt that arrives while one is
    being sent waits; if a newer one arrives too, the waiting one is dropped
    (last write wins) and reported as superseded.
    """

    def __init__(self):
        self.client = Seed42Client(config.CONTROL_API_URL)
        self.given_id = config.CONTROL_API_STREAM_ID or None
        self.stream_id = self.given_id
        self.cond = threading.Condition()
        self.busy = False
        self.latest = 0         # ticket of the newest send() call
        self.last_error = None

    def describe(self):
        return {"url": config.CONTROL_API_URL, "stream_id": self.stream_id,
                "given": bool(self.given_id), "last_error": self.last_error}

    def _ensure_stream(self):
        if self.client.token is None:
            self.client.login(config.CONTROL_API_EMAIL, config.CONTROL_API_PASSWORD)
        if self.stream_id:
            return
        sid, needs_poll = self.client.create_stream(config.CONTROL_API_INITIAL_PROMPT)
        if needs_poll and not self.client.wait_until_ready(sid):
            raise RuntimeError(f"stream {sid} was not ready after 30s")
        self.stream_id = sid
        log.info("stream ready: %s", sid)

    def send(self, prompt):
        """Returns {ok, code, attempts, ms, stream_id, error} and never raises."""
        with self.cond:
            self.latest += 1
            ticket = self.latest
            while self.busy:
                self.cond.wait()
            if ticket != self.latest:
                return {"ok": False, "superseded": True, "code": None, "attempts": 0, "ms": 0,
                        "stream_id": self.stream_id,
                        "error": "Replaced by a newer prompt before it was sent."}
            self.busy = True
        try:
            return self._send(prompt)
        finally:
            with self.cond:
                self.busy = False
                self.cond.notify_all()

    def _send(self, prompt):
        t0 = time.perf_counter()
        try:
            self._ensure_stream()
        except requests.ConnectionError:
            self.last_error = f"Can't reach the Control API at {config.CONTROL_API_URL}. Is the mock running?"
            return {"ok": False, "code": None, "attempts": 0, "ms": 0,
                    "stream_id": self.stream_id, "error": self.last_error}
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code == 401:
                self.client.token = None
            self.last_error = f"Couldn't log in or create a stream: {code}. {HINTS.get(code, '')}".strip()
            return {"ok": False, "code": code, "attempts": 0,
                    "ms": round((time.perf_counter() - t0) * 1000),
                    "stream_id": self.stream_id, "error": self.last_error}
        except Exception as e:
            self.last_error = f"Couldn't log in or create a stream: {e}"
            return {"ok": False, "code": None, "attempts": 0, "ms": 0,
                    "stream_id": self.stream_id, "error": self.last_error}

        sid = self.stream_id
        try:
            res = self.client.send_prompt(sid, prompt)
        except requests.RequestException as e:
            self.last_error = f"Request failed: {e}"
            return {"ok": False, "code": None, "attempts": 0, "ms": 0,
                    "stream_id": sid, "error": self.last_error}

        code, body = res["code"], res["body"]
        ok = code == 200
        error = None
        if code == 401:
            self.client.token = None    # log in again next time, same stream
            error = HINTS[401]
        elif code == 502 or (code == 404 and not is_not_ready(code, body)):
            # The guide: a 502 means the stream is gone and you need a new one
            what = "The stream has ended" if code == 502 else "The stream doesn't exist"
            if self.given_id:
                error = f"{what} ({code}). {NEXT_GIVEN}"
            else:
                self.stream_id = None
                error = f"{what} ({code}). {NEXT_OWN}"
        elif not ok:
            error = HINTS.get(code) or f"Unexpected {code}: {body}"
        self.last_error = error
        return {"ok": ok, "code": code, "attempts": res["attempts"],
                "ms": res["ms"], "stream_id": sid, "error": error}

    def reset(self):
        """End our own stream so the next prompt creates a new one. A stream
        seed42 gave us is left alone: ending it would stop their show."""
        with self.cond:
            while self.busy:
                self.cond.wait()
            if self.stream_id and not self.given_id:
                self.client.end_stream(self.stream_id)
                self.stream_id = None
            self.last_error = None


LINK = StreamLink()

Overwriting /content/seed42/server/app/control.py


**`server/app/jobs.py`**: run_window(): one window through the whole pipeline

In [15]:
%%writefile /content/seed42/server/app/jobs.py
"""Window pipeline plus the background worker for uploaded files.

run_window() takes one window through the whole pipeline:
features -> input sentence -> scene decision (scene.py) -> Control API.
The scene decides whether the LLM is called at all; a held scene re-sends
the same prompt.
File jobs and live sessions both use it, behind one lock, because the models
are shared and not safe to call from two threads at once.
"""
import logging
import os
import queue
import tempfile
import threading
import time
import uuid

import librosa

from . import config
from .control import LINK
from .features import analyse_window, energy_envelope
from .models import MODELS
from .scene import SCENE, find_gap

log = logging.getLogger("pipeline.jobs")

ANALYSIS_LOCK = threading.Lock()


def run_window(y_slice, index, t_start, t_end, use_llm=True, send_to_stream=True):
    """One window through the whole pipeline. Returns the record the page renders."""
    with ANALYSIS_LOCK:
        rec = analyse_window(y_slice, sr=config.SR)
    gap = find_gap(y_slice)

    rec.update({"index": index, "t_start": round(t_start, 2), "t_end": round(t_end, 2)})

    # The scene decides whether this window holds, adjusts, blends or replaces the prompt
    SCENE.step(rec, gap, use_llm)
    rec["prompt_words"] = len(rec["prompt"].split()) if rec["prompt"] else 0

    if send_to_stream and rec["prompt"]:
        rec["api"] = LINK.send(rec["prompt"])
    else:
        rec["api"] = {"ok": None, "code": None, "attempts": 0, "ms": 0,
                      "stream_id": None, "error": None, "skipped": True}
    return rec


class Job:
    def __init__(self, filename, path, window_sec, step_sec, use_llm, send_to_stream):
        self.id = uuid.uuid4().hex[:12]
        self.filename = filename
        self.path = path
        self.window_sec = window_sec
        self.step_sec = step_sec
        self.use_llm = use_llm
        self.send_to_stream = send_to_stream
        self.status = "queued"          # queued -> decoding -> waiting-for-models -> running -> done | failed | cancelled
        self.error = None
        self.duration = None
        self.envelope = None
        self.total_windows = None
        self.results = []
        self.cancelled = False
        self.created = time.time()

    def view(self, since=0):
        return {
            "id": self.id, "filename": self.filename, "status": self.status, "error": self.error,
            "duration": self.duration, "window_sec": self.window_sec, "step_sec": self.step_sec,
            "use_llm": self.use_llm, "send_to_stream": self.send_to_stream,
            "total_windows": self.total_windows, "done_windows": len(self.results),
            "envelope": self.envelope if since == 0 else None,
            "results": self.results[since:],
        }


JOBS = {}
_queue = queue.Queue()


def submit(job):
    JOBS[job.id] = job
    _queue.put(job.id)
    # Keep memory bounded: forget all but the 20 newest jobs
    for old in sorted(JOBS.values(), key=lambda j: j.created)[:-20]:
        JOBS.pop(old.id, None)
    return job


def window_starts(n_samples, window, step):
    """Window start offsets. A track shorter than one window is analysed whole."""
    if n_samples <= window:
        return [0]
    return list(range(0, n_samples - window + 1, step))


def _process(job):
    try:
        job.status = "decoding"
        y, sr = librosa.load(job.path, sr=config.SR, mono=True)
        job.duration = round(len(y) / sr, 2)
        job.envelope = energy_envelope(y, sr)
        window, step = int(job.window_sec * sr), int(job.step_sec * sr)
        starts = window_starts(len(y), window, step)
        job.total_windows = len(starts)

        if not MODELS.ready.is_set():
            job.status = "waiting-for-models"
            MODELS.ready.wait()

        job.status = "running"
        SCENE.start_track(job.filename)
        for i, start in enumerate(starts):
            if job.cancelled:
                job.status = "cancelled"
                return
            end = min(start + window, len(y))
            rec = run_window(y[start:end], i, start / sr, end / sr,
                             use_llm=job.use_llm, send_to_stream=job.send_to_stream)
            job.results.append(rec)
        job.status = "done"
    except Exception as e:
        log.exception("job %s failed", job.id)
        job.status = "failed"
        job.error = f"Couldn't analyse {job.filename}: {e}"
    finally:
        try:
            os.unlink(job.path)
        except OSError:
            pass


def _worker():
    while True:
        job_id = _queue.get()
        job = JOBS.get(job_id)
        if job and not job.cancelled:
            _process(job)
        elif job:
            job.status = "cancelled"


def start_worker():
    threading.Thread(target=_worker, name="job-worker", daemon=True).start()


def save_upload(fileobj, filename):
    suffix = os.path.splitext(filename or "")[1] or ".audio"
    fd, path = tempfile.mkstemp(suffix=suffix, prefix="seed42-")
    limit = config.MAX_UPLOAD_MB * 1024 * 1024
    written = 0
    with os.fdopen(fd, "wb") as out:
        while chunk := fileobj.read(1024 * 1024):
            written += len(chunk)
            if written > limit:
                out.close()
                os.unlink(path)
                raise ValueError(f"File is larger than {config.MAX_UPLOAD_MB} MB (MAX_UPLOAD_MB).")
            out.write(chunk)
    return path

Overwriting /content/seed42/server/app/jobs.py


**`mock/mock-control-api.py`**: seed42's mock Control API

In [16]:
%%writefile /content/seed42/mock/mock-control-api.py
#!/usr/bin/env python3
"""Mock of the seed42 Control API (/api/streams), for integration development.

Live streams are not a good build target: they cost real time to start, and an
unreliable one is indistinguishable from a bug in your own client. This
reproduces the contract in control-api.md closely enough to build the
whole request lifecycle against, on a laptop, for free.

    python mock-control-api.py
    # http://127.0.0.1:8787   token=test-token

Deliberately unfriendly where the real API is unfriendly, because a mock that
only serves the happy path teaches a client to crash in production:

  * a missing or wrong token is a real 401, not a warning
  * create is followed by a window of not-ready, so the retry loop is exercised
    rather than skipped
  * --not-ready 409 answers the other legal not-ready code, so a client that
    only handles 404 fails here rather than later
  * a PATCH carrying a creation-time field is answered 200, exactly as the real
    API would, and warns on stdout. That is the failure this file most exists
    to catch: it is invisible in the response, and in production it reloads the
    pipeline and stops the show for ~30 seconds.

Standard library only, on purpose: nobody should have to install anything to
test against this.
"""
import argparse
import json
import random
import secrets
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

# Fields fixed when the stream is created. Sending one in a PATCH forces a
# pipeline reload on the real API. Mirrors control-api.md section 3.
COLD_FIELDS = {
    "model_id", "width", "height", "acceleration",
    "lora_dict", "use_lcm_lora", "filters",
}
COLD_CONTROLNET_FIELDS = {"model_id", "preprocessor", "preprocessor_params"}

STREAMS = {}
LOCK = threading.Lock()
CFG = None


def now():
    return time.monotonic()


class Stream:
    """One mock stream. ready_at is when it stops answering not-ready;
    dies_at is set only under --unstable."""

    def __init__(self, params):
        self.id = "str_" + secrets.token_hex(8)
        self.params = dict(params or {})
        self.created = now()
        self.ready_at = self.created + CFG.ready_after
        self.dies_at = self.created + 45 if CFG.unstable else None
        self.patches = 0

    @property
    def state(self):
        if self.dies_at and now() >= self.dies_at:
            return "dead"
        return "ready" if now() >= self.ready_at else "starting"


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    # -- plumbing ---------------------------------------------------------
    def log_message(self, fmt, *args):
        print("  %s" % (fmt % args))

    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PATCH, DELETE, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, Authorization")
        self.end_headers()
        self.wfile.write(body)

    def _body(self):
        n = int(self.headers.get("Content-Length") or 0)
        if not n:
            return {}
        try:
            return json.loads(self.rfile.read(n) or b"{}")
        except json.JSONDecodeError:
            return None

    def _authed(self):
        """The real endpoint gates every verb but OPTIONS. Same here: a client
        that never sees a 401 in development meets its first one live."""
        header = self.headers.get("Authorization", "")
        if header.startswith("Bearer ") and header[7:] == CFG.token:
            return True
        self._send(401, {"error": "Not authenticated"})
        return False

    def _maybe_fail(self):
        if CFG.fail_rate and random.random() < CFG.fail_rate:
            self._send(502, {"error": "Upstream error",
                             "details": "injected by --fail-rate"})
            return True
        return False

    def _login(self):
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        if body.get("action") != "login":
            return self._send(400, {"error": "Unknown action"})
        if not body.get("email") or not body.get("password"):
            return self._send(400, {"error": "email and password required"})
        print("  login %s -> token" % body["email"])
        return self._send(200, {"ok": True, "token": CFG.token})

    def _not_ready(self):
        """The one retryable case. Both codes are legal; a client must treat
        them alike, so the mock can be told to answer either."""
        if CFG.not_ready == 409:
            return self._send(409, {"error": "Stream is not running", "status": "starting"})
        return self._send(404, {"error": "stream not ready"})

    # -- verbs ------------------------------------------------------------
    def do_OPTIONS(self):
        self._send(200, {})

    def do_GET(self):
        if not self._authed():
            return
        # Only meaningful after a 202. A client that polls unconditionally
        # should discover that here rather than in production.
        if not CFG.async_create:
            return self._send(404, {"error": "No status endpoint for this stream"})
        sid = (parse_qs(urlparse(self.path).query).get("id") or [None])[0]
        if not sid:
            return self._send(400, {"error": "id is required"})
        with LOCK:
            s = STREAMS.get(sid)
        if not s:
            return self._send(404, {"error": "Stream not found"})
        if s.state == "dead":
            return self._send(502, {"id": sid, "status": "failed",
                                    "error": "stream ended before it was ready"})
        if s.state == "ready":
            return self._send(200, {"id": sid, "status": "ready"})
        return self._send(200, {"id": sid, "status": "starting"})

    def do_POST(self):
        # /api/auth is the one call that carries no token, because it is how
        # you get one. Routed by path so a client written against the real
        # API -- login first, then streams -- runs here unchanged. Any
        # non-empty email and password are accepted; the answer is the
        # mock's own token.
        if urlparse(self.path).path.rstrip("/").endswith("/api/auth"):
            return self._login()
        if not self._authed():
            return
        if self._maybe_fail():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        s = Stream(body.get("params"))
        with LOCK:
            STREAMS[s.id] = s
        print("  created %s (ready in %.0fs%s)"
              % (s.id, CFG.ready_after, ", ends at 45s" if s.dies_at else ""))
        if CFG.async_create:
            return self._send(202, {"id": s.id, "status": "starting"})
        return self._send(201, {
            "id": s.id,
            "stream_key": "stk_" + secrets.token_hex(8),
            "output_stream_url": "rtmp://mock.local/live/" + secrets.token_hex(4),
            "params": s.params,
        })

    def do_PATCH(self):
        if not self._authed():
            return
        if self._maybe_fail():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        sid = body.get("id")
        if not sid:
            return self._send(400, {"error": "Stream id is required for PATCH"})
        with LOCK:
            s = STREAMS.get(sid)
        if not s:
            return self._send(404, {"error": "Stream not found"})
        if s.state == "dead":
            return self._send(502, {"error": "stream ended"})
        if s.state == "starting":
            return self._not_ready()

        params = body.get("params") or {}
        self._warn_cold(params)
        s.patches += 1
        s.params.update(params)
        print("  patch #%d on %s: %s" % (s.patches, sid, _summarise(params)))
        return self._send(200, {"ok": True, "id": sid})

    def do_DELETE(self):
        if not self._authed():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        sid = (body or {}).get("stream_id") or (body or {}).get("id")
        if not sid:
            return self._send(400, {"error": "Stream id is required for DELETE"})
        with LOCK:
            existed = STREAMS.pop(sid, None) is not None
        if CFG.unstable:
            # DELETE is allowed to fail. Clients must tolerate it, not retry it.
            return self._send(502, {"error": "Upstream error"})
        print("  deleted %s%s" % (sid, "" if existed else " (unknown)"))
        return self._send(200, {"ok": True, "id": sid})

    # -- the check the real API will not do for you -----------------------
    def _warn_cold(self, params):
        hits = sorted(COLD_FIELDS.intersection(params))
        for cn in params.get("controlnets") or []:
            if isinstance(cn, dict):
                hits += ["controlnets[].%s" % f
                         for f in sorted(COLD_CONTROLNET_FIELDS.intersection(cn))]
        if hits:
            print("\n  * COLD FIELD IN PATCH: %s" % ", ".join(hits))
            print("      Production would reload the pipeline (~30s) and the")
            print("      performance would stop. Send only hot fields.")
            print("      See control-api.md section 3.\n")


def _summarise(params):
    bits = []
    if "prompt" in params:
        p = str(params["prompt"])
        bits.append("prompt=%r" % (p[:40] + "..." if len(p) > 40 else p))
    for k in ("t_index_list", "guidance_scale", "delta", "seed", "num_inference_steps"):
        if k in params:
            bits.append("%s=%s" % (k, params[k]))
    cns = params.get("controlnets")
    if isinstance(cns, list):
        bits.append("cn=[%s]" % ",".join(
            str(c.get("conditioning_scale")) for c in cns if isinstance(c, dict)))
    return " ".join(bits) or "(empty)"


def main():
    global CFG
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--port", type=int, default=8787)
    ap.add_argument("--host", default="127.0.0.1",
                    help="bind address; use 0.0.0.0 inside Docker")
    ap.add_argument("--token", default="test-token")
    ap.add_argument("--not-ready", type=int, choices=(404, 409), default=404,
                    dest="not_ready",
                    help="which not-ready code to answer (both are legal; your "
                         "client must handle either)")
    ap.add_argument("--async-create", action="store_true", dest="async_create",
                    help="answer 202 on create and require the GET poll")
    ap.add_argument("--ready-after", type=float, default=4.0,
                    help="seconds a new stream answers not-ready (default 4)")
    ap.add_argument("--always-ready", action="store_true",
                    help="skip the not-ready window entirely")
    ap.add_argument("--unstable", action="store_true",
                    help="streams end ~45s after creation and DELETE fails")
    ap.add_argument("--fail-rate", type=float, default=0.0,
                    help="probability of a 502 on create/patch, e.g. 0.2")
    CFG = ap.parse_args()
    if CFG.always_ready:
        CFG.ready_after = 0.0

    # Line-buffer stdout. Python block-buffers when it is not a terminal, so
    # piping this to a file or running it from an IDE swallowed every warning
    # until the process exited -- including the cold-field warning, which is
    # the main reason to run this at all.
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except AttributeError:  # pragma: no cover - Python < 3.7
        pass

    srv = ThreadingHTTPServer((CFG.host, CFG.port), Handler)
    print("seed42 Control API mock")
    print("  http://127.0.0.1:%d   token=%s" % (CFG.port, CFG.token))
    print("  Authorization: Bearer %s" % CFG.token)
    print("  not-ready=%d%s%s"
          % (CFG.not_ready,
             "  async-create" if CFG.async_create else "",
             "  unstable" if CFG.unstable else ""))
    print("  contract: control-api.md      ctrl-c to stop\n")
    try:
        srv.serve_forever()
    except KeyboardInterrupt:
        print("\nstopped")


if __name__ == "__main__":
    main()

Overwriting /content/seed42/mock/mock-control-api.py


**`eval/compare.py`**: baseline vs refined evaluation

In [17]:
%%writefile /content/seed42/eval/compare.py
"""Baseline vs refined pipeline on a folder of tracks.

    docker compose run --rm eval                       # tracks in eval/tracks
    docker compose run --rm eval eval/tracks --judge   # also score prompts with the LLM

Baseline, the simplest end-to-end pipeline:
  features  Librosa only: tempo, energy phase, loudest band
  genre     always the most common label in labels.csv (majority class)
  prompt    a fixed template filled in from phase and band, no LLM
  scenes    none; the prompt is rebuilt every window

Refined, the pipeline the server runs:
  features  Madmom tempo, Essentia genre and mood, Whisper lyric themes
  prompt    written by the LLM (needs GROQ_API_KEY)
  scenes    scene tracking: hold, adjust, blend, replace (scene.py)

Both run on the same windows of the same tracks in one pass. Nothing is sent
to the Control API. Results go to eval/results/<time>/: windows.csv (one row
per window), summary.md (the comparison tables) and summary.json.

Ground truth is optional, in <tracks>/labels.csv:

    file,genre,bpm,boundaries
    song1.wav,rock,120,
    mix.mp3,,,95.5;201

genre and bpm score feature accuracy. boundaries (seconds, ;-separated) are
where one song ends and the next begins in a file of joined songs, and score
whether scene changes happen where the music changes.
"""
import argparse
import csv
import json
import re
import statistics
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
# The app package is at server/app in the repo and at /srv/app in the container
for base in (ROOT / "server", ROOT):
    if (base / "app").is_dir():
        sys.path.insert(0, str(base))
        break

import librosa  # noqa: E402

from app import config, prompt  # noqa: E402
from app.features import analyse_window  # noqa: E402
from app.jobs import window_starts  # noqa: E402
from app.models import MODELS  # noqa: E402
from app.scene import SceneTracker, find_gap  # noqa: E402

AUDIO = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif"}

# Baseline prompt: one phrase per energy phase, one per loudest band
PHASE_PHRASES = {
    "breakdown": "dim empty space, soft blue haze, slow drift",
    "groove": "night city street, warm amber light, steady pan",
    "build": "tunnel of rising light, magenta glow, camera pushing forward",
    "drop": "neon arena, white strobes, fast camera shake",
}
BAND_PHRASES = {
    "sub": "deep pulsing shapes",
    "mid": "glowing synth ribbons",
    "high": "glittering sparks",
}

SCENE_STARTS = {"new_scene", "new_track"}

JUDGE_PROMPT = (
    "You rate prompts for a real-time image stream that visualises music. "
    "Given a description of the music and a visual prompt, rate how well the "
    "prompt fits the music on a scale of 1 (no fit) to 5 (excellent fit), "
    "considering mood, energy, tempo and genre. Reply with the digit only."
)


def baseline_prompt(rec):
    return f"{PHASE_PHRASES[rec['phase']]}, {BAND_PHRASES[rec['band']]}"


def norm(s):
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())


def genre_match(label, predicted):
    """'metal' matches 'Rock---Heavy Metal'; 'hiphop' matches 'Hip Hop---Trap'."""
    return bool(label) and norm(label) in norm(predicted)


def tempo_hits(est, truth):
    """(acc1, acc2): within 4% of the true tempo; acc2 also accepts x2, x3, /2, /3."""
    if not est or not truth:
        return None, None
    acc1 = abs(est - truth) <= 0.04 * truth
    acc2 = any(abs(est - truth * f) <= 0.04 * truth * f for f in (1, 2, 3, 1 / 2, 1 / 3))
    return acc1, acc2


def judge(music, visual):
    """1-5 from the LLM, or None if it isn't available or answers oddly."""
    if prompt._client is None or not visual:
        return None
    try:
        reply = prompt._client.chat.completions.create(
            model=config.LLM_MODEL,
            messages=[{"role": "system", "content": JUDGE_PROMPT},
                      {"role": "user", "content": f"Music: {music}\nPrompt: {visual}"}],
            temperature=0,
            max_tokens=400,
            extra_body={"reasoning_effort": "low"},
        )
        m = re.search(r"[1-5]", reply.choices[0].message.content or "")
        return int(m.group()) if m else None
    except Exception as e:
        print(f"    judge failed: {e}")
        return None


def read_labels(path):
    if not path or not path.exists():
        return {}
    with open(path, newline="", encoding="utf-8") as f:
        rows = {}
        for r in csv.DictReader(f):
            bounds = [float(b) for b in (r.get("boundaries") or "").split(";") if b.strip()]
            rows[r["file"].strip()] = {
                "genre": (r.get("genre") or "").strip(),
                "bpm": float(r["bpm"]) if (r.get("bpm") or "").strip() else None,
                "boundaries": bounds,
            }
        return rows


def run_track(path, label, args):
    """Every window of one track through both pipelines. Returns the window rows."""
    y, sr = librosa.load(path, sr=config.SR, mono=True)
    window, step = int(args.window * sr), int(args.step * sr)
    scenes = SceneTracker()          # a fresh scene tracker for every track
    rows, prev_base, prev_ref = [], None, None

    for i, start in enumerate(window_starts(len(y), window, step)):
        end = min(start + window, len(y))
        rec = analyse_window(y[start:end], sr=sr)
        rec.update({"index": i, "t_start": round(start / sr, 2), "t_end": round(end / sr, 2)})
        base = baseline_prompt(rec)
        scenes.step(rec, find_gap(y[start:end]), use_llm=not args.no_llm)
        ref = rec["prompt"]
        decision = rec["scene"]["decision"]
        t = rec["timings"]

        row = {
            "track": path.name, "window": i, "t_start": rec["t_start"], "t_end": rec["t_end"],
            "energy": rec["energy"], "phase": rec["phase"], "band": rec["band"],
            "true_bpm": label.get("bpm"), "base_bpm": rec["librosa_bpm"], "ref_bpm": rec["bpm"],
            "ref_bpm_source": rec["bpm_source"],
            "true_genre": label.get("genre", ""), "ref_genre": rec["primary_genre"],
            "ref_mood": rec["mood"], "ref_themes": rec["lyric_themes"],
            "music_description": rec["llm_input"],
            "base_prompt": base, "ref_prompt": ref or "", "ref_prompt_source": rec["prompt_source"],
            "ref_decision": decision,
            "base_changed": prev_base is not None and base != prev_base,
            "ref_changed": prev_ref is not None and ref is not None and ref != prev_ref,
            "base_scene_start": prev_base is not None and base != prev_base,
            "ref_scene_start": decision in SCENE_STARTS,
            "base_ms": t.get("librosa", 0) + t.get("bands", 0),
            "ref_ms": rec["analysis_ms"] + rec["llm_ms"],
            "base_judge": None, "ref_judge": None,
        }
        if args.judge:
            row["base_judge"] = judge(rec["llm_input"], base)
            row["ref_judge"] = judge(rec["llm_input"], ref)
        rows.append(row)
        prev_base, prev_ref = base, (ref if ref is not None else prev_ref)
        print(f"  window {i + 1}: {decision:<9} {rec['bpm']:>6} BPM  {rec['primary_genre']}")
    return rows, len(y) / sr


def boundary_scores(rows, boundaries, starts_key, window, step):
    """(boundaries caught, total, scene changes not near a boundary).

    A boundary counts as caught if a scene starts in a window ending between
    the boundary and one window plus one step after it, because scene tracking
    blends first and confirms on the next window.
    """
    starts = [r["t_end"] for r in rows if r[starts_key]]
    near = lambda t, b: b <= t <= b + window + step  # noqa: E731
    caught = sum(any(near(t, b) for t in starts) for b in boundaries)
    spurious = sum(not any(near(t, b) for b in boundaries) for t in starts)
    return caught, len(boundaries), spurious


def mean(xs):
    xs = [x for x in xs if x is not None]
    return statistics.mean(xs) if xs else None


def pct(xs):
    xs = [x for x in xs if x is not None]
    return 100 * sum(xs) / len(xs) if xs else None


def fmt(v, unit="", digits=1):
    return "n/a" if v is None else f"{v:.{digits}f}{unit}"


def summarise(rows, durations, labels, args):
    audible = [r for r in rows if r["energy"] >= config.SCENE_SILENCE_RMS]
    minutes = sum(durations.values()) / 60
    s = {}

    # Tempo
    tempo = [r for r in audible if r["true_bpm"]]
    for side in ("base", "ref"):
        hits = [tempo_hits(r[f"{side}_bpm"], r["true_bpm"]) for r in tempo]
        s[f"{side}_tempo_acc1"] = pct(h[0] for h in hits)
        s[f"{side}_tempo_acc2"] = pct(h[1] for h in hits)
    s["tempo_windows"] = len(tempo)

    # Genre: the baseline always answers the most common label
    genre = [r for r in audible if r["true_genre"]]
    track_genres = [lab["genre"] for lab in labels.values() if lab["genre"]]
    majority = Counter(norm(g) for g in track_genres).most_common(1)[0][0] if track_genres else None
    s["majority_genre"] = majority
    s["base_genre_acc"] = pct(norm(r["true_genre"]) == majority for r in genre) if majority else None
    s["ref_genre_acc"] = pct(genre_match(r["true_genre"], r["ref_genre"]) for r in genre)         if any(r["ref_genre"] != "unavailable" for r in genre) else None
    s["genre_windows"] = len(genre)

    # Prompts and scenes
    for side in ("base", "ref"):
        s[f"{side}_changes_per_min"] = sum(r[f"{side}_changed"] for r in rows) / minutes if minutes else None
        s[f"{side}_scenes_per_min"] = sum(r[f"{side}_scene_start"] for r in rows) / minutes if minutes else None
        caught = total = spurious = 0
        for track, lab in labels.items():
            if lab["boundaries"]:
                c, t, sp = boundary_scores([r for r in rows if r["track"] == track], lab["boundaries"],
                                           f"{side}_scene_start", args.window, args.step)
                caught, total, spurious = caught + c, total + t, spurious + sp
        s[f"{side}_boundary_recall"] = 100 * caught / total if total else None
        s[f"{side}_boundary_spurious"] = spurious if total else None
        s["boundaries"] = total
        s[f"{side}_judge"] = mean(r[f"{side}_judge"] for r in rows)
        s[f"{side}_ms"] = mean(r[f"{side}_ms"] for r in rows)

    s["llm_prompts"] = sum(r["ref_prompt_source"] == "llm" for r in rows)
    return s


def write_markdown(path, s, meta):
    # Scene tracking only changes scenes when the LLM writes the new prompt
    no_llm = "needs the LLM; scenes are held without it" if not s["llm_prompts"] else ""
    rows = [
        ("Tempo accuracy, within 4%", fmt(s["base_tempo_acc1"], "%"), fmt(s["ref_tempo_acc1"], "%"),
         f"{s['tempo_windows']} labelled windows"),
        ("Tempo accuracy, allowing x2 / /2", fmt(s["base_tempo_acc2"], "%"), fmt(s["ref_tempo_acc2"], "%"), ""),
        ("Genre accuracy", fmt(s["base_genre_acc"], "%"), fmt(s["ref_genre_acc"], "%"),
         f"{s['genre_windows']} labelled windows; baseline always answers '{s['majority_genre']}'"
         if s["majority_genre"] else "needs genre labels"),
        ("Track changes caught", fmt(s["base_boundary_recall"], "%"), fmt(s["ref_boundary_recall"], "%"),
         no_llm or (f"{s['boundaries']} labelled boundaries" if s["boundaries"] else "needs boundary labels")),
        ("Scene changes away from a track change", fmt(s["base_boundary_spurious"], "", 0),
         fmt(s["ref_boundary_spurious"], "", 0), no_llm or "lower is better"),
        ("Scene changes per minute", fmt(s["base_scenes_per_min"], "", 2), fmt(s["ref_scenes_per_min"], "", 2), no_llm),
        ("Prompt changes per minute", fmt(s["base_changes_per_min"], "", 2), fmt(s["ref_changes_per_min"], "", 2),
         no_llm or "includes scene adjustments"),
        ("Prompt fit, LLM judge (1-5)", fmt(s["base_judge"], "", 2), fmt(s["ref_judge"], "", 2),
         "needs --judge and GROQ_API_KEY" if s["base_judge"] is None else ""),
        ("Time per window", fmt(s["base_ms"], " ms", 0), fmt(s["ref_ms"], " ms", 0), "analysis + LLM"),
    ]
    lines = [
        "# Baseline vs refined pipeline", "",
        f"Run {meta['time']}, {meta['tracks']} tracks, {meta['windows']} windows, "
        f"window {meta['window']} s, step {meta['step']} s.", "",
        f"Mode: {meta['mode']}. Models: {meta['models']}. "
        f"LLM: {meta['llm']} ({s['llm_prompts']} prompts written by it).", "",
        "| Metric | Baseline | Refined | Notes |", "|---|---|---|---|",
        *[f"| {a} | {b} | {c} | {d} |" for a, b, c, d in rows], "",
        "Baseline: Librosa tempo, energy and band; majority-class genre; template prompt; no scene tracking.",
        "Refined: Madmom, Essentia and Whisper features; LLM prompt; scene tracking.", "",
        "The judge sees the refined pipeline's description of the music, and the refined prompt was "
        "written from that same description, so treat the judge score as a guide, not proof.",
    ]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("tracks", nargs="?", default=str(HERE / "tracks"), help="folder of audio files")
    ap.add_argument("--labels", help="ground truth CSV (default: labels.csv in the tracks folder)")
    ap.add_argument("--window", type=float, default=config.DEFAULT_WINDOW_SEC, help="window length, seconds")
    ap.add_argument("--step", type=float, default=config.DEFAULT_STEP_SEC, help="step between windows, seconds")
    ap.add_argument("--judge", action="store_true", help="score every prompt with the LLM (2 calls per window)")
    ap.add_argument("--no-llm", action="store_true", help="refined pipeline without the LLM")
    ap.add_argument("--out", default=str(HERE / "results"), help="where to write results")
    args = ap.parse_args()

    tracks_dir = Path(args.tracks)
    files = sorted(p for p in tracks_dir.glob("*") if p.suffix.lower() in AUDIO)
    if not files:
        sys.exit(f"No audio files in {tracks_dir}. Put .wav, .mp3 or .flac files there.")
    labels = read_labels(Path(args.labels) if args.labels else tracks_dir / "labels.csv")
    unknown = set(labels) - {p.name for p in files}
    if unknown:
        print(f"labels.csv lists files that aren't in {tracks_dir}: {', '.join(sorted(unknown))}")
    if not prompt.llm_enabled() or args.no_llm:
        print("Warning: no LLM, so the refined pipeline holds one scene per track and its "
              "scene results mean little. Set GROQ_API_KEY in .env for a fair comparison.")
    if args.judge and not prompt.llm_enabled():
        print("--judge needs GROQ_API_KEY; prompts won't be scored.")

    print(f"Loading models ({config.PIPELINE_MODE} mode)...")
    MODELS.load()
    models = ", ".join(f"{k} {v['state']}" for k, v in MODELS.status.items())
    print(models)

    rows, durations = [], {}
    for n, path in enumerate(files, 1):
        print(f"[{n}/{len(files)}] {path.name}")
        try:
            track_rows, durations[path.name] = run_track(path, labels.get(path.name, {}), args)
            rows += track_rows
        except Exception as e:
            print(f"  skipped: {e}")
    if not rows:
        sys.exit("No track could be analysed.")

    out = Path(args.out) / datetime.now().strftime("%Y%m%d-%H%M%S")
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "windows.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    s = summarise(rows, durations, {k: v for k, v in labels.items() if k in durations}, args)
    meta = {"time": time.strftime("%Y-%m-%d %H:%M"), "tracks": len(durations), "windows": len(rows),
            "window": args.window, "step": args.step, "mode": config.PIPELINE_MODE, "models": models,
            "llm": "off" if args.no_llm else (config.LLM_MODEL if prompt.llm_enabled() else "no key, fallback prompt")}
    (out / "summary.json").write_text(json.dumps({"meta": meta, "metrics": s}, indent=2), encoding="utf-8")
    print("\n" + write_markdown(out / "summary.md", s, meta))
    print(f"\nResults in {out}")


if __name__ == "__main__":
    main()

Overwriting /content/seed42/eval/compare.py


---
## 3. Settings, mock server and stream
### 3.1 Settings
The notebook's `.env`: the same values as `.env.example`. They must be set before the server code is imported.

In [18]:
import os, sys, logging, warnings

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get(GROQ_SECRET_NAME) or ''
except Exception as e:
    print('No Groq key from Colab secrets; prompts will use the fallback.', e)

# From .env.example
os.environ["LLM_MODEL"] = "openai/gpt-oss-120b"
os.environ["MAX_PROMPT_WORDS"] = "40"
os.environ["PIPELINE_MODE"] = "full"
os.environ["WHISPER_SIZE"] = "tiny"
os.environ["N_THEMES"] = "4"
os.environ["CONTROL_API_EMAIL"] = "test@example.com"
os.environ["CONTROL_API_PASSWORD"] = "anything"
os.environ["CONTROL_API_STREAM_ID"] = ""
os.environ["SCENE_EVOLVE_AT"] = "0.2"
os.environ["SCENE_CHANGE_AT"] = "0.4"
os.environ["SCENE_GAP_SEC"] = "1.5"
os.environ["DEFAULT_WINDOW_SEC"] = "10"
os.environ["DEFAULT_STEP_SEC"] = "5"
# For the real seed42 API instead of the mock, set these (and your own login above):
# os.environ['CONTROL_API_URL'] = 'https://seed42.app'

os.chdir('/content/seed42/server')
sys.path.insert(0, '/content/seed42/server')
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)   # quiet; the reports below print what matters

from app import config
print(f"Mode {config.PIPELINE_MODE} | LLM {config.LLM_MODEL if config.GROQ_API_KEY else 'off (no key)'} | "
      f"window {config.DEFAULT_WINDOW_SEC:g} s, step {config.DEFAULT_STEP_SEC:g} s | "
      f"scene thresholds {config.SCENE_EVOLVE_AT} / {config.SCENE_CHANGE_AT}")

Mode full | LLM openai/gpt-oss-120b | window 10 s, step 5 s | scene thresholds 0.2 / 0.4


### 3.2 Mock Server Connection
Starts seed42's mock Control API once, with `MOCK_FLAGS`, and keeps it running until the cleanup cell at the end.

In [20]:
!pkill -f mock-control-api.py


In [21]:
import subprocess, time, signal

MOCK_LOG = "/content/seed42/server.log"
mock_proc = None

def start_mock(flags=()):
    global mock_proc
    stop_mock()
    mock_proc = subprocess.Popen([sys.executable, "-u", "/content/seed42/mock/mock-control-api.py", *flags],
                                 stdout=open(MOCK_LOG, "w"), stderr=subprocess.STDOUT,
                                 start_new_session=True)
    time.sleep(1.5)
    if mock_proc.poll() is not None:
        print(open(MOCK_LOG).read())
        raise RuntimeError("The mock didn't start; its output is above.")
    print("Mock started", " ".join(flags) or "(normal mode)")

def stop_mock():
    global mock_proc
    if mock_proc and mock_proc.poll() is None:
        os.killpg(mock_proc.pid, signal.SIGTERM)
        mock_proc.wait()
    mock_proc = None

def mock_log(tail=20):
    print("".join(open(MOCK_LOG).readlines()[-tail:]))

start_mock(MOCK_FLAGS)

Mock started (normal mode)


In [22]:
# Connect: log in, create a stream and send the opening prompt, as the server does on its first window
from app.control import LINK
r = LINK.send(config.CONTROL_API_INITIAL_PROMPT)
if r["ok"]:
    print(f"Stream ready: {r['stream_id']} ({r['code']} in {r['ms']} ms, {r['attempts']} {'try' if r['attempts'] == 1 else 'tries'})")
else:
    print(f"Not connected: {r['error']}")

Stream ready: str_c483835251a6562a (200 in 4010 ms, 3 tries)


---
## 4. Load models
`MODELS.load()` is what the server runs at startup. Model files download once and are reused.
Expected: `madmom`, `essentia`, `mood` and `whisper` all **ready**.

In [23]:
from app.models import MODELS
MODELS.load()
for name, s in MODELS.status.items():
    print(f"{name:9s} {s['state']:8s} {s['detail']}")

madmom    ready    
essentia  ready    400 genres
mood      ready    happy, sad, relaxed, aggressive, party
whisper   ready    tiny on cpu


In [24]:
# Playlist: the files that exist, copied into eval/tracks (where eval/compare.py looks).
# Entries can be file names (looked up in AUDIO_DIR) or full paths.
# eval/compare.py needs ordinary extensions, so ".mp3.mpeg" is copied as ".mp3".
import shutil
TRACKS_DIR = "/content/seed42/eval/tracks"
TRACKS = []
for entry in AUDIO_PLAYLIST:
    src = entry if os.path.isabs(entry) else os.path.join(AUDIO_DIR, entry)
    if not os.path.exists(src):
        print(f" Track not found, skipped: {src}")
        continue
    name = os.path.basename(src)
    dst = os.path.join(TRACKS_DIR, name[:-5] if name.lower().endswith(".mpeg") else name)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    TRACKS.append(dst)
print(f"{len(TRACKS)} of {len(AUDIO_PLAYLIST)} track(s) ready")


3 of 3 track(s) ready


---
## 5. Test on one track (sliding window)
Runs the feature extractor (`features.analyse_window`) over the first track, without the LLM or the stream:
- **Window:** 20 seconds of audio analysed at a time
- **Step:** the window moves forward 5 seconds each time

On *Play House – Marijuana* (139 s) this gives **24 windows**. Results are saved to `eval/results/one-track-20s-5s.csv`.

In [25]:
import csv, json, time, librosa
from app.features import analyse_window
from app.jobs import window_starts

def run_one_track(path, window_sec=20, step_sec=5):
    y, sr = librosa.load(path, sr=config.SR, mono=True)
    window, step = int(window_sec * sr), int(step_sec * sr)
    print("=" * 110)
    print(" CONTINUOUS STREAM RUNNER (Librosa + Madmom + Essentia + Whisper)")
    print(f" Window: {window_sec}s | Step: every {step_sec}s | File: {os.path.basename(path)}")
    print("=" * 110)
    print(f"{'Time':>8} | {'BPM':>6} | {'Energy':>7} | {'Phase':<9} | {'Top 2 genres':<34} | {'Mood':<18} | Proc")
    print("-" * 110)
    rows, t_run = [], time.time()
    for start in window_starts(len(y), window, step):
        end = min(start + window, len(y))
        rec = analyse_window(y[start:end], sr=sr)
        rec["t_end"] = round(end / sr, 1)
        rows.append(rec)
        print(f"{rec['t_end']:>7.1f}s | {rec['bpm']:>6.1f} | {rec['energy']:>7.4f} | {rec['phase']:<9} | "
              f"{rec['top_2_genres'][:34]:<34} | {rec['mood'][:18]:<18} | {rec['analysis_ms']:>5}ms")
    print("-" * 110)
    print(f"Done: {len(rows)} windows in {time.time() - t_run:.1f} s")
    return rows

if TRACKS:
    output_stream = run_one_track(TRACKS[0])
    keys = ["t_end", "bpm", "bpm_source", "madmom_bpm", "librosa_bpm", "energy", "phase", "band",
            "primary_genre", "secondary_genre", "mood", "lyric_themes", "analysis_ms"]
    with open("/content/seed42/eval/results/one-track-20s-5s.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader(); w.writerows(output_stream)
else:
    output_stream = []
    print("No track available.")

 CONTINUOUS STREAM RUNNER (Librosa + Madmom + Essentia + Whisper)
 Window: 20s | Step: every 5s | File: 003-2235411-Jackson Frost-Fashionable Progressive House.mp3
    Time |    BPM |  Energy | Phase     | Top 2 genres                       | Mood               | Proc
--------------------------------------------------------------------------------------------------------------
   20.0s |  125.0 |  0.0861 | breakdown | Electronic---Progressive House, El | party              | 14641ms
   25.0s |  125.0 |  0.0973 | breakdown | Electronic---Progressive House, El | party              | 10245ms
   30.0s |  125.0 |  0.1010 | breakdown | Electronic---Progressive House, El | party, relaxed     |  9113ms
   35.0s |  125.0 |  0.1325 | breakdown | Electronic---Progressive House, El | party, relaxed     |  7876ms
   40.0s |  125.0 |  0.2086 | groove    | Electronic---Progressive House, El | party, relaxed     | 10248ms
   45.0s |  125.0 |  0.2651 | groove    | Electronic---Progressive House, El | p

### Inspect the results
The first window in full. `llm_input` is the sentence the LLM receives.

In [26]:
if output_stream:
    print(json.dumps(output_stream[0], indent=2, default=float))

{
  "bpm": 125.0,
  "bpm_source": "Madmom neural",
  "madmom_bpm": 125.0,
  "librosa_bpm": 123.0,
  "beat_count": 41,
  "energy": 0.0861,
  "top_2_genres": "Electronic---Progressive House, Electronic---Deep House",
  "genre_scores": [
    0.314,
    0.269
  ],
  "primary_genre": "Electronic---Progressive House",
  "secondary_genre": "Electronic---Deep House",
  "mood": "party",
  "mood_scores": {
    "happy": 0.063,
    "sad": 0.103,
    "relaxed": 0.45,
    "aggressive": 0.138,
    "party": 0.907
  },
  "lyric_themes": "instrumental",
  "timings": {
    "librosa": 1952,
    "madmom": 6793,
    "essentia": 3798,
    "whisper": 2069,
    "bands": 21
  },
  "phase": "breakdown",
  "phase_label": "Low Energy / Breakdown Phase",
  "band": "sub",
  "band_label": "Low-End / Sub-Bass Focus (20 Hz - 120 Hz)",
  "llm_input": "A party mood, Low Energy / Breakdown Phase 125.0 BPM Progressive House atmosphere, featuring Low-End / Sub-Bass Focus with instrumental themes.",
  "analysis_ms": 14641,
 

---
## 6. Playlist engine
Every track goes through exactly what an upload on the test page does: windows of `DEFAULT_WINDOW_SEC`
every `DEFAULT_STEP_SEC`, and per window `jobs.run_window()`: features → sentence → **scene decision** →
prompt (LLM only when the scene changes) → Control API.

The **scene decision** compares the window with the music the current scene was written for
(tempo, genre, mood, energy, band → a change score from 0 to 1):

| Change score | Decision | Prompt |
|---|---|---|
| below `SCENE_EVOLVE_AT` | **hold** | the same prompt is re-sent; no LLM call |
| `SCENE_EVOLVE_AT` to `SCENE_CHANGE_AT` | **evolve** | the LLM changes up to three phrases |
| `SCENE_CHANGE_AT` or more | **blend** | a halfway prompt; the next window confirms (**new scene**) or **reverts** |
| a silence gap (≥ 1.5 s) | **blend → new track** | a likely track change |

A new track always blends from the previous scene, then starts its own.

In [27]:
from app.jobs import run_window
from app.scene import SCENE

def mmss(s):
    m, s = divmod(int(round(s)), 60)
    return f"{m:02d}:{s:02d}"

def report(rec, track_no, total, name, update_no):
    sc, api = rec["scene"], rec["api"]
    scores = ", ".join(f"{k} {v:.2f}" for k, v in sorted(rec.get("mood_scores", {}).items(), key=lambda x: -x[1])[:3])
    print("=" * 80)
    print(f" LIVE AUDIO VIBE STREAM - UPDATE #{update_no}")
    print(f" Track {track_no:02d}/{total:02d} ({name}) | [{mmss(rec['t_start'])} -> {mmss(rec['t_end'])}]")
    print("=" * 80)
    print(f" • Tempo                   : {rec['bpm']:.1f} BPM ({rec['bpm_source']})")
    print(f" • Primary genre           : {rec['primary_genre']}")
    print(f" • Secondary genre         : {rec['secondary_genre']}")
    print(f" • Mood                    : {rec['mood']}" + (f"  ({scores})" if scores else ""))
    print(f" • Energy (RMS)            : {rec['energy']:.3f} ({rec['phase_label']})")
    print(f" • Lyric themes            : {rec['lyric_themes']}")
    print(f" • Dominant frequency band : {rec['band_label']}")
    print(" " + "-" * 78)
    print(f"  INPUT SENT TO LLM: \"{rec['llm_input']}\"")
    print(" " + "-" * 78)
    score = "" if sc["score"] is None else f"change {sc['score']:.2f}: " + ", ".join(f"{k} {v:g}" for k, v in sc["parts"].items())
    print(f"  SCENE DECISION : {sc['decision'].upper()}" + (f"  ({score})" if score else ""))
    print(f"                   {sc['note']}")
    if rec["prompt"]:
        print(f"  VISUAL PROMPT ({rec['prompt_words']}/{config.MAX_PROMPT_WORDS} words, {rec['prompt_source']}):")
        print(f"    {rec['prompt']}")
    if rec.get("llm_error"):
        print(f"  LLM: {rec['llm_error']}")
    if api.get("skipped"):
        print("  CONTROL API    : not sent")
    elif api["ok"]:
        tries = f"{api['attempts']} {'try' if api['attempts'] == 1 else 'tries'}"
        print(f"  CONTROL API    : {api['code']} in {api['ms']} ms ({tries}) -> stream {api['stream_id']}")
    else:
        print(f"  CONTROL API    : {api['code'] or 'no response'} - {api['error']}")
    print("=" * 80 + "\n")

def run_playlist(use_llm=False, send=False):
    """Plays every track back to back, like a DJ set."""
    print(f"\n[Playlist] {len(TRACKS)} track(s) | LLM {'ON' if use_llm else 'OFF'} | send {'ON' if send else 'OFF'} | "
          f"window {config.DEFAULT_WINDOW_SEC:g} s every {config.DEFAULT_STEP_SEC:g} s\n")
    records = []
    for track_no, path in enumerate(TRACKS, 1):
        name = os.path.basename(path)
        y, sr = librosa.load(path, sr=config.SR, mono=True)
        window, step = int(config.DEFAULT_WINDOW_SEC * sr), int(config.DEFAULT_STEP_SEC * sr)
        SCENE.start_track(name)
        for i, start in enumerate(window_starts(len(y), window, step)):
            end = min(start + window, len(y))
            rec = run_window(y[start:end], i, start / sr, end / sr, use_llm=use_llm, send_to_stream=send)
            rec["track"] = name
            records.append(rec)
            report(rec, track_no, len(TRACKS), name, i + 1)
    return records

print("Engine defined.")

Engine defined.


### Run the playlist (no LLM, nothing sent)
Shows the features and input sentence for every window. Without the LLM the scene can't change, so decisions show **wait** or **hold**.

In [28]:
records_no_llm = run_playlist(use_llm=False, send=False)


[Playlist] 3 track(s) | LLM OFF | send OFF | window 10 s every 5 s

 LIVE AUDIO VIBE STREAM - UPDATE #1
 Track 01/03 (003-2235411-Jackson Frost-Fashionable Progressive House.mp3) | [00:00 -> 00:10]
 • Tempo                   : 125.0 BPM (Madmom neural)
 • Primary genre           : Electronic---House
 • Secondary genre         : Electronic---Deep House
 • Mood                    : party  (party 0.82, relaxed 0.43, aggressive 0.20)
 • Energy (RMS)            : 0.073 (Low Energy / Breakdown Phase)
 • Lyric themes            : instrumental
 • Dominant frequency band : Low-End / Sub-Bass Focus (20 Hz - 120 Hz)
 ------------------------------------------------------------------------------
  INPUT SENT TO LLM: "A party mood, Low Energy / Breakdown Phase 125.0 BPM House atmosphere, featuring Low-End / Sub-Bass Focus with instrumental themes."
 ------------------------------------------------------------------------------
  SCENE DECISION : WAIT
                   Prompt generation is off and

---
## 7. Full pipeline: LLM and Control API
### 7.1 LLM
`openai/gpt-oss-120b` on Groq, called by `prompt.py` only when the scene decision needs a new or revised prompt.
Prompts are limited to **40 words** (`MAX_PROMPT_WORDS`) because SDXL-Turbo reads only about 77 tokens:
1. **Instructions:** one line of 8–12 short comma-separated phrases (scene → materials → lighting → atmosphere → glow → camera), no sentences, numbers or genre names.
2. **Token cap:** `max_tokens=400` with `reasoning_effort="low"`.
3. **Python trim:** anything over the limit is cut back to the last complete phrase.

Temperature: **0.7** for a new scene, **0.4** to evolve, **0.6** to blend. If the LLM fails, the current scene is kept.

In [29]:
from app.prompt import llm_enabled
print(f"Groq client ready (model: {config.LLM_MODEL})." if llm_enabled() else
      "Warning: Groq not configured. The scene can't change and prompts use the fallback.")

Groq client ready (model: openai/gpt-oss-120b).


### 7.2 Full pipeline
Audio → sentence → scene decision → LLM → prompt → stream, for every window of every track. Records are saved to `eval/results/notebook-<time>.csv`.

In [30]:
from datetime import datetime
records = run_playlist(use_llm=True, send=True)

FIELDS = ["track", "index", "t_start", "t_end", "bpm", "bpm_source", "energy", "phase", "band", "primary_genre",
          "secondary_genre", "mood", "lyric_themes", "llm_input", "prompt", "prompt_source", "analysis_ms", "llm_ms"]
out = f"/content/seed42/eval/results/notebook-{datetime.now():%Y%m%d-%H%M%S}.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(FIELDS + ["decision", "change_score", "api_code", "api_attempts"])
    for r in records:
        w.writerow([r.get(k) for k in FIELDS] + [r["scene"]["decision"], r["scene"]["score"],
                                                 r["api"].get("code"), r["api"].get("attempts")])
decisions = {}
for r in records:
    decisions[r["scene"]["decision"]] = decisions.get(r["scene"]["decision"], 0) + 1
sent = [r["api"] for r in records if not r["api"].get("skipped")]
print(f"{len(records)} windows | decisions {decisions} | sent ok {sum(a['ok'] for a in sent)}/{len(sent)}")
print("Saved", out)


[Playlist] 3 track(s) | LLM ON | send ON | window 10 s every 5 s

 LIVE AUDIO VIBE STREAM - UPDATE #1
 Track 01/03 (003-2235411-Jackson Frost-Fashionable Progressive House.mp3) | [00:00 -> 00:10]
 • Tempo                   : 125.0 BPM (Madmom neural)
 • Primary genre           : Electronic---House
 • Secondary genre         : Electronic---Deep House
 • Mood                    : party  (party 0.82, relaxed 0.43, aggressive 0.20)
 • Energy (RMS)            : 0.073 (Low Energy / Breakdown Phase)
 • Lyric themes            : instrumental
 • Dominant frequency band : Low-End / Sub-Bass Focus (20 Hz - 120 Hz)
 ------------------------------------------------------------------------------
  INPUT SENT TO LLM: "A party mood, Low Energy / Breakdown Phase 125.0 BPM House atmosphere, featuring Low-End / Sub-Bass Focus with instrumental themes."
 ------------------------------------------------------------------------------
  SCENE DECISION : INITIAL
                   First audible window, so it

In [31]:
mock_log(25)   # what the mock received

  "PATCH /api/streams HTTP/1.1" 200 -
  patch #165 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #166 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #167 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #168 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #169 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #170 on str_c483835251a6562a: prompt='rooftop lounge with people swaying, plus...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #171 on str_c483835251a6562a: prompt='rooftop lounge, plush velvet sofas, glos...'
  "PATCH /api/streams HTTP/1.1" 200 -
  patch #172 on str_c483835251a6562a: prompt='lounge with f

---
## 8. Evaluation: baseline vs refined
Runs `eval/compare.py` on the same tracks, exactly as `docker compose run --rm eval` does (nothing is sent to the stream).
Add `--judge` to score prompts with the LLM. There is no labelled reference set yet, so accuracy rows show `n/a`
until `eval/tracks/labels.csv` is added.

In [32]:
!cd /content/seed42 && python eval/compare.py eval/tracks

Loading models (full mode)...
2026-10-19 06:14:22.478970: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.13/dist-packages/cv2/../../lib64:
2026-10-19 06:14:22.479087: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
[   INFO   ] MusicExtractorSVM: no classifier models were configured by default
2026-10-19 06:14:23.124184: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-19 06:14:23.124623: W tensorflow/stream_executor/platf

In [33]:
import glob
from IPython.display import Markdown, display
latest = sorted(p for p in glob.glob("/content/seed42/eval/results/*") if os.path.isdir(p))[-1]
display(Markdown(open(f"{latest}/summary.md", encoding="utf-8").read()))

# Baseline vs refined pipeline

Run 2026-10-19 06:34, 3 tracks, 175 windows, window 10.0 s, step 5.0 s.

Mode: full. Models: madmom ready, essentia ready, mood ready, whisper ready. LLM: openai/gpt-oss-120b (52 prompts written by it).

| Metric | Baseline | Refined | Notes |
|---|---|---|---|
| Tempo accuracy, within 4% | n/a | n/a | 0 labelled windows |
| Tempo accuracy, allowing x2 / /2 | n/a | n/a |  |
| Genre accuracy | n/a | n/a | needs genre labels |
| Track changes caught | n/a | n/a | needs boundary labels |
| Scene changes away from a track change | n/a | n/a | lower is better |
| Scene changes per minute | 1.20 | 0.40 |  |
| Prompt changes per minute | 1.20 | 3.40 | includes scene adjustments |
| Prompt fit, LLM judge (1-5) | n/a | n/a | needs --judge and GROQ_API_KEY |
| Time per window | 53 ms | 6964 ms | analysis + LLM |

Baseline: Librosa tempo, energy and band; majority-class genre; template prompt; no scene tracking.
Refined: Madmom, Essentia and Whisper features; LLM prompt; scene tracking.

The judge sees the refined pipeline's description of the music, and the refined prompt was written from that same description, so treat the judge score as a guide, not proof.


---
## 9. Control API failure tests (optional)
seed42's guide asks for one run against each of these before live access. Each test restarts the mock with its
flags, creates a fresh stream and sends a few prompts through the server's client.

| Flags | Expected |
|---|---|
| `--not-ready 404 --ready-after 20` | `404` three times per send, *Still not ready after 3 tries*, then `200` |
| `--not-ready 409 --ready-after 20` | The same with `409` |
| `--async-create` | `POST 202`, `GET ?id=…` polls until ready, then `PATCH 200` |
| `--unstable` | `200` until 45 s, then `502` *The stream has ended*, a new stream, `200` again |

In [34]:
def api_test(flags, sends=4, gap=3.0):
    print("=" * 80, "\n mock", " ".join(flags))
    start_mock(flags)
    LINK.reset()                       # forget the old stream; the next send creates one
    for n in range(sends):
        r = LINK.send(f"api test prompt {n + 1}")
        print(f"  send {n + 1}: ok={r['ok']} code={r['code']} attempts={r['attempts']} {r['ms']} ms  {r['error'] or ''}")
        time.sleep(gap)
    mock_log(14)

RUN_API_TESTS = False   # True to run all four (about 3 minutes; --unstable waits 45 s)
if RUN_API_TESTS:
    api_test(["--not-ready", "404", "--ready-after", "20"])
    api_test(["--not-ready", "409", "--ready-after", "20"])
    api_test(["--async-create"])
    api_test(["--unstable"], sends=6, gap=10)
    start_mock(MOCK_FLAGS)
else:
    print("Skipped (RUN_API_TESTS = False).")

Skipped (RUN_API_TESTS = False).


---
## 10. Cleanup
Ends the stream and stops the mock. Run at the end only.

In [35]:
LINK.reset()
stop_mock()
print("Mock API stopped.")

Mock API stopped.
